# Eukaryotic toehold switch — ON/OFF report (single switch + AND)

Built from `and_eu_report_spec.md` in this folder. Measures each candidate's real,
Boltzmann-sampled ON/OFF behaviour through `FoldEngine` only (no NUPACK, no MFE-only
shortcuts) and decomposes every sampled state into three independent signals —
**`trigger_bound`** (paired to an external strand), **`stem_closed`** (the intended OFF
hairpin actually closed), **`misfolded`** (paired to neither) — rather than collapsing
them into one "open" bit. That collapse is exactly what let an earlier version of this
project's AND construct read 80-90% "open" with **zero triggers present**, purely from
intramolecular cross-hairpin misfolding (see Traps, bottom of the spec).

Reuses `fuse`/`measure`/`load_ranked_designs`/`motif_self_bind_score` from
`_switch_construct.py` (migrated out of `toehold_and_eu_test.ipynb` in an earlier commit
on this branch) and the parallel workers + `stem_closed_fraction`/`misfolded_fraction`/
`_frac_paired_to_external`/`pick_fake_design` from `_joint_state_parallel.py` — nothing
here reimplements either module.

**Reading `OPEN_THRESHOLD = 0.70`:** numbers in this notebook are *not* directly
comparable to `toehold_and_eu_test.ipynb`'s reports, which used `0.80`.


## Setup

In [1]:
# Put the shared _fixtures.py (notebooks/ root) and this folder itself
# (_switch_construct.py / _joint_state_parallel.py) on sys.path, then fx.bootstrap()
# adds <repo>/src so `import engine...` resolves. No Django, no worker, no pipeline.
import sys, pathlib

for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_fixtures.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break
for _base in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_base / "_joint_state_parallel.py").exists():
        if str(_base) not in sys.path:
            sys.path.insert(0, str(_base))
        break

import _fixtures as fx
fx.bootstrap()


engine package root: /Users/zeev/Projects/cernal_software/src


PosixPath('/Users/zeev/Projects/cernal_software/src')

## Parameters — every tunable in one place (and_eu_report_spec.md Step 1)

In [2]:
# ============================================================================
# PARAMETERS. If a later cell needs a new threshold, it belongs HERE, not inline.
# ============================================================================

# ---- data source -----------------------------------------------------------
CSV_PATH = (
    "/Users/zeev/Projects/NucSyn_Public_Platform_0.1.1/nucsyn-platform-work/"
    "output_T2/targets/sele/basic_switch/results_ranked.csv"
)
# NucSyn runner export, topology euk_open_5p_kozak_after_stem, already sorted
# best-first by postgen_global_rank. Printed on both reports.

# Column-name map load_ranked_designs() needs -- a REQUIRED parameter there, not a
# module default (_switch_construct.py's own docstring: a second copy of a column
# map living inside the shared module is the same drift problem CLAUDE.md sec 1 warns
# about, one level down).
COLUMNS = {
    "switch": "switch_strand",
    "trigger": "trigger_seq",
    "domains": "switch_domain_breakdown_json",
    "rank": "postgen_global_rank",
    "trigger_start": "trigger_start_pos",
    "trigger_end": "trigger_end_pos",
}
# CSV columns carried through into design["metrics"] unchanged -- never recomputed
# here (CLAUDE.md sec 5: a value already computed upstream is read, not re-derived).
CARRY_METRICS = ("sim_score", "mcc", "switch_toehold_unpaired_prob_mean", "energy_trigger_mfe")

# fuse()'s own required parameters (and_eu_report_spec.md Step 2's AND topology): the
# 5' half keeps everything before its OWN kozak starts, the 3' half keeps everything
# after its OWN prefix ends.
CUT_5P_BEFORE = "kozak"
KEEP_3P_AFTER = "prefix"

# ---- folding -----------------------------------------------------------------
TEMPERATURE_C = 37.0  # folding temperature (deg C), recorded on every energy below

# ---- single-switch screen -----------------------------------------------------
SINGLE_POOL_N = 300    # ranked designs loaded for the single-switch screen
SINGLE_REPORT_N = 20   # how many reach the Single Switch report

# ---- AND grid -------------------------------------------------------------------
AND_GRID_N = 20   # top-N singles (by single-switch ON/OFF ratio) fed into the N x N grid
AND_REPORT_N = 20 # how many AND pairs reach the AND report

# ---- sampling --------------------------------------------------------------------
N_SAMPLES = 1200        # Boltzmann samples drawn per state
OPEN_THRESHOLD = 0.70   # fraction of footprint bound externally to call a state "open"
# 0.70, not the predecessor notebook's 0.80 -- numbers here are NOT directly
# comparable to that notebook's reports (and_eu_report_spec.md Step 5).

# ---- fake-trigger counts -- two separate parameters, same default, different reason -
SINGLE_N_FAKES = 10
# Distinct fake triggers checked per single-switch candidate, averaged into its OFF
# term -- "10 different fake triggers... the mean will be the result". A single switch
# has ONE footprint, so there is only one pool to draw this many fakes from.
AND_N_FAKES = 10
# Distinct fakes drawn PER SIDE for each AND pair -- 10 for the A-side states, 10 for
# the B-side states (20 fake-involving draws per pair, two INDEPENDENT rotations).
# A separate parameter from SINGLE_N_FAKES not because the count differs (both default
# to 10) but because AND has two footprints screened against, drawn from, and reported
# against separately -- changing one side's count must never silently change the
# other's.

# ---- fake-trigger selection (Trap 2 / Trap 3 fixes) -------------------------------
FAKE_MAX_SEQ_OVERLAP_NT = 8  # max complementary run (wobbles included) a fake may have
WOBBLE_MAX_FRACTION = 0.50   # max share of a matched run that may be G.U wobble

# ---- AND spacer --------------------------------------------------------------------
SPACER_VARIANTS = ("no_spacer", "spacer")  # both are run and reported separately
SPACER_MIN_LENGTH = 15
SPACER_MAX_LENGTH = 20
SPACER_SEED = 1                   # literal seed -- never wall-clock (CLAUDE.md sec 6)
SPACER_CANDIDATE_POOL_SIZE = 150  # pool generate_spacer_candidates() draws from

# ---- optional homopolymer filter ----------------------------------------------------
FILTER_HOMOPOLYMER_RUN = False   # exclude toeholds with a run > MAX_HOMOPOLYMER_RUN
MAX_HOMOPOLYMER_RUN = 5

# ---- execution ------------------------------------------------------------------------
MAX_WORKERS = 6   # process-pool width -- one pool call per BATCH, never one per job

# ---- snapshot --------------------------------------------------------------------------
SNAPSHOT_TO_LOAD = (
    "/Users/zeev/Projects/cernal_software/var/reports/and_eu_report_snapshot_20261004T203534.json"
)
# Set to a snapshot filename (or an absolute path) to skip every heavy cell below and
# restore single_report_rows/and_report_rows from a previous run instead (the restore
# + the n_stem_closed/n_misfolded backfill both happen in the cell right after the
# "Part 1" heading, not here -- see that cell's own docstring). None means a normal
# full run -- every heavy cell actually runs instead. Pointed at a real snapshot from
# a completed full run (2026-10-04) for the 2026-10-05 report-fix pass: per that
# session's own explicit instruction, no full re-run (the ~8 min 300-candidate screen
# + 20x20 AND grid) for this pass, iteration AND the final deliverable both render
# from this snapshot (+ the cheap per-row backfill) instead.


In [ ]:
# One FoldEngine for the WHOLE notebook (single-switch part and AND part both reuse
# it) -- one cache, one temperature (CLAUDE.md sec 1/sec 5: FoldEngine is the only
# place a gate family folds, and exactly one instance exists per run).
folder = fx.fold_engine(real=True, temperature=TEMPERATURE_C)

# Publish that one engine to the parallel-worker module, which folds through it
# instead of importing ViennaRNA itself. The pools fork, so every worker inherits
# THIS instance -- must happen before any pool starts.
from _joint_state_parallel import use_fold_engine
use_fold_engine(folder)
print(f"FoldEngine published to the worker module: {folder.versions()}")

from _switch_construct import fuse, load_ranked_designs, measure
from _joint_state_parallel import (
    AndDecompJob,
    CrossDomainJob,
    SingleDecompJob,
    generate_spacer_candidates,
    pick_fake_design,
    pick_optimized_spacer,
    run_parallel_and_decomp,
    run_parallel_cross_domain,
    run_parallel_single_decomp,
)
from engine.sequences import longest_homopolymer

import datetime, html, json, time
from pathlib import Path


## Plotting + report CSS helpers

In [ ]:
# ==================================================================================
# Plotting helpers -- same conventions as toehold_and_eu_test.ipynb (2D structure via
# FoldEngine.layout_coordinates, a real base-pair-probability heatmap, a segmented
# layout bar for the domain diagram). Nothing here re-folds anything: every sequence
# plotted was already folded by measure()/the decomposition jobs earlier in this run,
# so folder.mfe/.base_pair_probabilities calls below are cache hits.
# ==================================================================================
import base64
from io import BytesIO

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import numpy as np

_COLOR_A = "#4C6E8C"   # hairpin A / 5' half / single switch's one hairpin
_COLOR_B = "#B5507E"   # hairpin B / 3' half (AND only)
_KOZAK_COLOR = "#2E9E56"
_DEFAULT_FILL = "#C7D0CA"   # a base outside every known domain span (e.g. the trigger
                             # strand itself, which has no domain breakdown)
_TRIGGER_COLOR = "#D1495B"  # 2026-10-05 fix (single item 10): the real trigger strand
                             # on the ON 2D structure plot used to fall through to
                             # _DEFAULT_FILL (no domain_fill entry covered it at all) --
                             # it now gets its own distinct color, separate from every
                             # domain hue below.
_SPACER_COLOR = "#3F8F8C"   # 2026-10-05 fix (AND item 14): "spacer" was not a key in
                             # either domain palette, so it fell through to
                             # _DEFAULT_FILL and was invisible on both the AND layout
                             # bar and the structure plots that show it. One shared
                             # teal, not per-side, since the spacer itself isn't part
                             # of either hairpin.

# 2026-10-05 fix (single item 9 / AND item 14): "loop" used to be the exact same
# color as _DEFAULT_FILL (#C7D0CA) -- meaning it had no visible color on the 2D
# structure plot at all -- and "prefix" was nearly indistinguishable from both
# _DEFAULT_FILL and "loop". Both now get their own distinct hues, chosen to also
# stay clear of every other domain color already in use (toehold/stem_up/stem_down/
# kozak/exp_gene) and of _TRIGGER_COLOR/_SPACER_COLOR above.
_DOMAIN_COLOR_A = {
    "prefix": "#C9A227", "toehold": "#4C6E8C", "stem_up": "#7E97AC",
    "loop": "#8E6AA8", "stem_down": "#7E97AC", "kozak": "#2E9E56", "exp_gene": "#B5760E",
    "spacer": _SPACER_COLOR,
}
# AND needs a doubled palette so hairpin 1 and 2 are visually distinguishable
# (and_eu_report_spec.md Step 4) -- same domain names, a second, B-tinted hue each.
_DOMAIN_COLOR_B = {
    "prefix": "#B98B5E", "toehold": "#B5507E", "stem_up": "#C98CA8",
    "loop": "#6B4E79", "stem_down": "#C98CA8", "kozak": "#2E9E56", "exp_gene": "#8E6A3A",
    "spacer": _SPACER_COLOR,
}

# 2026-10-05 fix (single item 11 / AND item 12): domains whose coloring/segment is
# dropped from the RENDERED layout bar and 2D structure plots -- the payload itself
# stays a real domain in the underlying data (fuse()/measure() untouched), this only
# decides what structure_png/layout_bar_png are actually asked to draw. Shared here
# (not re-declared per images cell) so the single-switch and AND images cells can't
# quietly drift to two different payload-domain lists.
_PAYLOAD_DOMAINS = ("exp_gene", "opt_exp_gene")


def _png_from_fig(fig):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", facecolor="white")
    plt.close(fig)
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("ascii")


def _domain_fill_from_domains(domains, palette, label_suffix=""):
    """Builds the ``domain_fill`` list :func:`structure_png` wants -- (start, end,
    color, label) -- from a plain ``{name: (start, end)}`` domain-breakdown dict and a
    name->color palette (2026-10-05 feedback fix #6: real per-domain fill color on the
    2D structure plot, not just the flat ``_DEFAULT_FILL`` every base used to get).
    ``label_suffix`` disambiguates two hairpins sharing domain names in the AND report
    (" (A)"/" (B)") so the legend shows both, not one overwriting the other.
    """
    return [
        (s, e, palette.get(name, _DEFAULT_FILL), f"{name}{label_suffix}")
        for name, (s, e) in domains.items()
        if e > s
    ]


def structure_png(sequence, structure, title, spans=(), domain_fill=(), figsize=(3.6, 3.2), visible_len=None):
    """One folded structure as a base64 PNG. Coordinates from
    ``FoldEngine.layout_coordinates`` (the shared 2D layout).

    ``domain_fill``: ``(start, end, color, label)`` tuples -- the REAL per-domain fill
    color for every base in that span (fix #6), with a legend built from the unique
    ``(label, color)`` pairs actually present. A base outside every span keeps the flat
    ``_DEFAULT_FILL`` (e.g. a trigger strand, which has no domain breakdown of its own).

    ``spans``: ``(start, end, color)`` tuples -- the ORIGINAL ring-highlight behaviour,
    unchanged and drawn on top of the fill (e.g. the trigger-bound footprint). The two
    are independent: domain_fill decides what's inside the dot, spans decides the ring
    around it, so passing both together highlights a footprint without losing which
    domain each base belongs to.

    ``visible_len``: 2026-10-05 fix (AND item 7) -- when set, only positions
    ``0..visible_len`` of ``sequence`` (the switch/fused construct's OWN bases) are
    drawn at all; any appended trigger strand (or any base at/after this index) is
    simply never scattered. A base pair with ONE partner outside ``visible_len`` --
    i.e. a switch base paired to a trigger -- draws no connecting line (the trigger
    end does not exist on this plot) but the switch-side base gets a distinguishing
    ring instead (reusing the existing ``spans`` ring mechanism, a dangling "bound
    externally" stub rather than a line to nowhere). A pair with BOTH partners
    outside ``visible_len`` (entirely among hidden trigger bases) is skipped
    entirely. ``None`` (the default) draws everything, unchanged from before.
    """
    points = folder.layout_coordinates(structure)
    xs = [x for x, _ in points]
    ys = [y for _, y in points]
    _vis = len(sequence) if visible_len is None else visible_len

    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    ax.plot(xs[:_vis], ys[:_vis], "-", color="#B9C2BC", linewidth=1.1, zorder=1)
    stack = []
    _cross_bound = set()  # switch-side bases paired to a now-hidden trigger base
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            i_vis, j_vis = i < _vis, j < _vis
            if i_vis and j_vis:
                ax.plot([xs[i], xs[j]], [ys[i], ys[j]], "-", color="#8899AA",
                         linewidth=0.9, zorder=1, alpha=0.7)
            elif i_vis or j_vis:
                _cross_bound.add(i if i_vis else j)
            # else: neither partner visible -- entirely among hidden bases, skipped.
    for i in range(_vis):
        fill = next((c for s, e, c, _l in domain_fill if s <= i < e), _DEFAULT_FILL)
        ring = next((c for s, e, c in spans if s <= i < e), None)
        if ring is None and i in _cross_bound:
            # Dangling-stub indicator (AND item 7): this switch base is really paired
            # to a trigger base we are deliberately not drawing -- a distinct ring,
            # same color convention as the trigger itself, says "bound externally,
            # partner hidden" instead of silently looking unpaired.
            ring = _TRIGGER_COLOR
        ax.scatter(xs[i], ys[i], s=24 if ring else 13, color=fill,
                    edgecolors=ring or "#666666", linewidths=1.3 if ring else 0.3,
                    zorder=3 if ring else 2)
    ax.text(xs[0], ys[0] + 8, "5'", fontsize=7, ha="center", color="#345")
    ax.text(xs[_vis - 1], ys[_vis - 1] + 8, "3'", fontsize=7, ha="center", color="#345")
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")
    if domain_fill:
        seen = []
        for _s, _e, _c, _l in domain_fill:
            if (_l, _c) not in seen:
                seen.append((_l, _c))
        handles = [
            mlines.Line2D([0], [0], marker="o", color="w", markerfacecolor=_color,
                          markeredgecolor="#666666", markersize=6, linewidth=0, label=_label)
            for _label, _color in seen
        ]
        ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.03),
                  ncol=min(len(handles), 4), fontsize=5.6, frameon=False,
                  handletextpad=0.3, columnspacing=0.8)
    plt.tight_layout()
    return _png_from_fig(fig)


def bpp_heatmap_png(sequence, matrix, title, boundaries=(), figsize=(3.4, 3.1)):
    """Base-pair-probability heatmap -- labelled colourbar, numbered (position) axis
    ticks.

    2026-10-05 fix (single item 3 / AND item 3): this used to switch to per-base
    LETTER tick labels once the sequence was short enough (<=70nt) for one tick per
    base to fit at all -- but "fit" there only meant "there is room for one tick per
    base", not "legible": the letters rendered at ~4.2pt and were illegibly small in
    practice (user feedback). Always uses the numeric-position-tick branch now,
    regardless of sequence length -- the same numbered-axis heatmap this project used
    before the letter-tick behaviour was added.

    ``boundaries``: ``(position, color)`` pairs -- one dashed CROSSHAIR (a horizontal
    AND a vertical dotted line, not a box) per real strand junction actually present in
    this matrix's ``&``-joined frame, each in its own strand's color. Replaces the
    previous ``plt.Rectangle`` outline entirely (2026-10-05 feedback fix #7: "not a
    dashed box, not a box at all" -- marking a 1-D position in a 2-D matrix needs both
    axes drawn to be legible, hence a line on each axis rather than a single line).
    A matrix with no second strand (an alone/OFF fold) passes no boundaries, since
    there is no junction to mark.
    """
    arr = np.array(matrix)
    fig, ax = plt.subplots(figsize=figsize, dpi=130)
    im = ax.imshow(arr, cmap="magma", vmin=0.0, vmax=1.0, origin="upper")
    for position, color in boundaries:
        ax.axhline(position - 0.5, color=color, linewidth=1.1, linestyle=":")
        ax.axvline(position - 0.5, color=color, linewidth=1.1, linestyle=":")
    n = len(sequence)
    step = max(1, n // 12)
    ticks = list(range(0, n, step))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks], fontsize=6.5, rotation=90)
    ax.set_yticklabels([str(t) for t in ticks], fontsize=6.5)
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(n - 0.5, -0.5)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("P(paired)", fontsize=7.5)
    cbar.ax.tick_params(labelsize=6.5)
    ax.set_title(title, fontsize=9, fontweight="bold")
    plt.tight_layout()
    return _png_from_fig(fig)


def layout_bar_png(segments, total_len, title):
    """Segmented domain-layout bar. `segments` is a list of
    (start, label, length, color, sequence) -- one bar for a single switch's one
    hairpin, or both hairpins of an AND construct in one fused-frame bar."""
    fig, ax = plt.subplots(figsize=(9.6, 2.0), dpi=130)
    for start, label, length, color, seq in sorted(segments, key=lambda s: s[0]):
        ax.barh(0, length, left=start, height=0.6, color=color, edgecolor="white", linewidth=1.2)
        if length / total_len > 0.02:
            ax.text(start + length / 2, 0, str(length), ha="center", va="center",
                    fontsize=7.5, fontweight="bold", color="white")
            ax.text(start + length / 2, -0.55, label, ha="center", va="top",
                    fontsize=6.6, color="#46524B")
            ax.text(start + length / 2, -1.05, seq, ha="center", va="top",
                    fontsize=5.0, family="monospace", color="#6B7870")
    ax.set_xlim(0, total_len)
    ax.set_ylim(-1.65, 0.5)
    ax.axis("off")
    ax.set_title(title, fontsize=10, fontweight="bold", loc="left")
    plt.tight_layout()
    return _png_from_fig(fig)


_REPORT_CSS = """
  :root {
    --scr-ground: #F6F8F4; --scr-surface: #FFFFFF; --scr-surface-2: #EEF2EA;
    --scr-ink: #131A16; --scr-ink-2: #46524B; --scr-ink-3: #75837B; --scr-line: #DCE4D8;
    --scr-accent: #2E9E56; --scr-accent-glow: #E4F7EA; --scr-accent-ink: #146134;
    --scr-structural: #4C6E8C; --scr-structural-bg: #E9EFF4;
    --scr-amber: #B5760E; --scr-amber-bg: #FBF0DD;
    --scr-shadow: 0 1px 2px rgba(19,26,22,0.04), 0 8px 24px -12px rgba(19,26,22,0.12);
  }
  @media (prefers-color-scheme: dark) {
    :root:not([data-theme="light"]) {
      --scr-ground: #0E1512; --scr-surface: #141D18; --scr-surface-2: #1A2620;
      --scr-ink: #E9F2EC; --scr-ink-2: #AEC0B6; --scr-ink-3: #7C8D83; --scr-line: #26362D;
      --scr-accent: #4CE07A; --scr-accent-glow: #163524; --scr-accent-ink: #7DF0A2;
      --scr-structural: #8FB3D6; --scr-structural-bg: #1B2A38;
      --scr-amber: #E0AA4C; --scr-amber-bg: #362B14;
      --scr-shadow: 0 1px 2px rgba(0,0,0,0.3), 0 8px 24px -12px rgba(0,0,0,0.5);
    }
  }
  * { box-sizing: border-box; }
  body.scr { margin: 0; background: var(--scr-ground); color: var(--scr-ink);
    font-family: "IBM Plex Sans", system-ui, sans-serif; padding: 0 20px; }
  .scr-mono { font-family: "IBM Plex Mono", ui-monospace, monospace; font-variant-numeric: tabular-nums; }
  .scr-wrap { max-width: 1040px; margin: 0 auto; padding-block: 44px 72px; }
  .scr-wrap h1 { font-size: clamp(26px, 4vw, 34px); font-weight: 700; line-height: 1.15;
    letter-spacing: -0.01em; margin: 0 0 8px; text-wrap: balance; }
  .scr-eyebrow { font-family: "IBM Plex Mono", monospace; font-size: 12px; letter-spacing: 0.08em;
    text-transform: uppercase; color: var(--scr-structural); margin: 0 0 10px; }
  .scr-dek { font-size: 15px; line-height: 1.6; color: var(--scr-ink-2); max-width: 78ch; margin: 0 0 32px; }
  .scr-wrap section { margin-bottom: 40px; }
  .scr-wrap h2 { font-size: 20px; font-weight: 700; letter-spacing: -0.01em; color: var(--scr-ink);
    margin: 0 0 12px; padding-bottom: 8px; border-bottom: 2px solid var(--scr-line); }
  .scr-wrap h3 { font-family: "IBM Plex Mono", monospace; font-size: 14px; font-weight: 600;
    color: var(--scr-accent-ink); background: var(--scr-accent-glow); display: inline-block;
    padding: 4px 10px; border-radius: 6px; margin: 0 0 14px; }
  .scr-p { font-size: 13.5px; line-height: 1.7; color: var(--scr-ink-2); margin: 0 0 10px; max-width: 82ch; }
  .scr-callout { background: var(--scr-structural-bg); border-left: 3px solid var(--scr-structural);
    border-radius: 4px; padding: 12px 16px; font-size: 13px; line-height: 1.6;
    color: var(--scr-ink-2); margin: 0 0 12px; }
  .scr-table-wrap { overflow-x: auto; border: 1px solid var(--scr-line); border-radius: 10px;
    background: var(--scr-surface); box-shadow: var(--scr-shadow); margin-bottom: 8px; }
  .scr-table-wrap table { width: 100%; border-collapse: collapse; font-size: 12px; }
  .scr-table-summary table { min-width: 980px; }
  .scr-table-wrap th { text-align: left; font-size: 10px; letter-spacing: 0.05em; text-transform: uppercase;
    color: var(--scr-ink-3); font-weight: 600; padding: 8px 10px; border-bottom: 1px solid var(--scr-line);
    background: var(--scr-surface-2); white-space: nowrap; }
  .scr-table-wrap th.num { text-align: right; }
  .scr-table-wrap td { padding: 7px 10px; border-bottom: 1px solid var(--scr-line); color: var(--scr-ink-2); }
  .scr-table-wrap tr:last-child td { border-bottom: none; }
  /* Every numeric measurement column carries class="num" on BOTH its <th> and every
     <td> (fix #1): a cell that mixes a percentage with an nt-count and a sample count
     (e.g. "100.0% (27.0/27 nt, n=1200)") is still ONE td with ONE class, so the whole
     compound string right-aligns and lines up with its column as a single unit rather
     than only the leading digits doing so. tabular-nums keeps digit widths equal so a
     column of different-length numbers still lines up vertically on its decimal point. */
  .scr-table-wrap td.num { text-align: right; font-family: "IBM Plex Mono", monospace;
    white-space: nowrap; font-variant-numeric: tabular-nums; }
  /* 2026-10-05 fix (single item 5 / AND item 1): the k/N-pct-mean compound cells
     (e.g. "1200/1200" on one line, "100.0% of samples · mean 99.0%" on a second,
     shorter-font line) are too long to fit a fixed-width decomp-table column on one
     nowrap line -- the secondary <small> line is allowed to wrap, independent of its
     parent td's own nowrap, so it drops to a second visual line inside its own cell
     instead of overflowing into the neighbour column (which `table-layout:fixed`,
     added this round so <colgroup> widths are actually honoured, would otherwise
     make simply bleed past the cell's own fixed boundary). */
  .scr-table-wrap td.num small { white-space: normal; display: block; }
  /* 2026-10-05 fix (regression found verifying the PDF render): a long free-text
     explanatory cell (e.g. the single report's "+fake trigger" row when 0/N fakes
     qualified) used to carry class="num" purely to get the right colspan -- which
     also inherited td.num's white-space:nowrap and ran the whole sentence off the
     fixed-width table/page edge with nothing to reveal the clipped tail in a static
     PDF (no scrollbar there). This class is for a free-text cell that needs normal
     left-aligned wrapping instead, independent of table-layout:fixed. */
  .scr-table-wrap td.scr-cell-note { text-align: left; font-family: inherit;
    white-space: normal; font-variant-numeric: normal; }
  .scr-candidate { background: var(--scr-surface); border: 1px solid var(--scr-line); border-radius: 12px;
    padding: 20px 22px; box-shadow: var(--scr-shadow); margin-bottom: 18px; }
  .scr-candidate-head { display: flex; align-items: baseline; justify-content: space-between;
    gap: 12px; margin-bottom: 12px; flex-wrap: wrap; }
  .scr-candidate-head h3 { margin: 0; }
  .scr-candidate-id { font-family: "IBM Plex Mono", monospace; font-size: 12px; color: var(--scr-ink-3); }
  .scr-img { max-width: 100%; height: auto; margin-bottom: 10px; }
  .scr-structure-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(240px, 1fr));
    gap: 14px; margin-top: 10px; margin-bottom: 14px; }
  .scr-structure-grid figure { margin: 0; background: var(--scr-surface-2); border-radius: 8px;
    padding: 6px; text-align: center; }
  .scr-structure-grid img { width: 100%; height: auto; }
  .scr-structure-grid figcaption { font-size: 11px; color: var(--scr-ink-3); margin-top: 4px; }
  .scr-seq-block { font-size: 11px; line-height: 1.8; margin-bottom: 4px; word-break: break-all; }
  .scr-seq-block .label { display: inline-block; width: 120px; color: var(--scr-ink-3);
    font-family: "IBM Plex Mono", monospace; }
  footer.scr-footer { border-top: 1px solid var(--scr-line); padding-top: 18px; font-size: 11.5px;
    color: var(--scr-ink-3); }
  @media print {
    :root, :root:not([data-theme="light"]) {
      --scr-ground: #FFFFFF; --scr-surface: #FFFFFF; --scr-surface-2: #F3F5F1;
      --scr-ink: #131A16; --scr-ink-2: #3A443E; --scr-ink-3: #5C6660; --scr-line: #C9D2C4;
      --scr-shadow: none;
    }
    body.scr { padding: 0 12px; }
    .scr-candidate, .scr-table-wrap { box-shadow: none; }
    .scr-candidate-head, .scr-seq-block, .scr-structure-grid figure, tr { break-inside: avoid; }
    thead { display: table-header-group; }
    /* The k-of-N decomposition tables (fix #5) are wider than the old bare-percent
       ones -- "1200/1200 (99.0%)" vs "99.0%" -- so print gets a tighter font/padding
       to keep every column on the page instead of running the last one(s) off the
       landscape A4 edge (render_pdf.py renders @media print directly via Chromium). */
    .scr-table-wrap table { font-size: 8px; }
    .scr-table-wrap td, .scr-table-wrap th { padding: 3px 4px; }
    .scr-table-wrap th small { font-size: 6.8px; font-weight: 400; text-transform: none;
      letter-spacing: 0; display: block; }
    /* 2026-10-05 fix (single item 3 / AND item 1): the energy table is only ever two
       columns with plenty of room -- exempt it from the tight squeeze above (rule
       ORDER, not just specificity, is what wins here: same selector weight as
       .scr-table-wrap, placed after it in this same @media block). */
    .scr-table-energy table { font-size: 10.5px; }
    .scr-table-energy td, .scr-table-energy th { padding: 5px 8px; }
    /* 2026-10-05 fix (AND item 1): the AND decomp table (7 data columns + state) needs
       its own smaller base size even in print, on top of the explicit <colgroup>
       widths the render cell gives it -- the generic 8px above is still too wide for
       7 k/N-pct-mean compound cells to fit a landscape page without this. */
    .scr-table-and-decomp table { font-size: 7px; }
    .scr-table-and-decomp td, .scr-table-and-decomp th { padding: 2px 3px; }
  }
"""


def _num(value, places=2, suffix=""):
    """'--' for a measurement that is not there (CLAUDE.md sec 3), never a 0."""
    return "--" if value is None else f"{value:.{places}f}{suffix}"


def _pct(value):
    return "--" if value is None else f"{value:.1%}"


def _nt(fraction, length):
    # 2026-10-05 fix (item K): show the denominator -- "{x}/{length} nt", not a bare
    # nt-count with no span to compare it against (inconsistent with _pct_nt_n's own
    # "27.0/27 nt" pattern just below).
    return "--" if fraction is None else f"{fraction * length:.1f}/{length} nt"


def _nt_pct(fraction, length):
    """Like ``_nt`` but with the mean % spelled out next to the nt-count (2026-10-05
    fix, single item 1 / AND item 1) -- e.g. "6.8/18 nt (37.8%)", not just the raw
    nt-count with no percentage next to it at all. Lets a reader directly compare the
    shown % against OPEN_THRESHOLD without doing the division themselves. Used
    specifically for FAKES-related nt displays (every other ``_nt`` call -- alone/
    real/+A/+B's own nt-counts -- is left as a bare nt-count, unchanged)."""
    return "--" if fraction is None else f"{fraction * length:.1f}/{length} nt ({fraction:.1%})"


def _kofn(n_open, n_samples):
    # None guard (2026-10-05): a few callers now pass state.get("n_..._open") for a
    # state dict that may predate these exact-count fields -- "--" is CLAUDE.md sec 3's
    # "a metric you could not compute is None, never 0" rendered, not "None/1200".
    return "--" if n_open is None or n_samples is None else f"{n_open}/{n_samples}"


def _pct_nt_n(pct_value, mean_fraction, length, n_samples):
    """ON/OFF display (fix #2): a bare percentage on its own doesn't say what it means
    or how it was measured, so every ON/OFF percentage in this report is shown with (a)
    how many Boltzmann samples it was drawn from, and (b) the underlying nt-count, not
    just the fraction -- e.g. "100.0% (27.0/27 nt, n=1200)". ``pct_value`` is the
    open-FREQUENCY (k/N samples crossing OPEN_THRESHOLD) shown as the headline percent;
    ``mean_fraction`` is the companion CONTINUOUS mean (mean_trigger_bound for ON,
    mean_stem_closed for OFF) that the nt-count is computed from -- a different
    quantity from the headline percent, deliberately: the headline answers "how often
    did this clear the open bar", the nt-count answers "on average, how much of the
    span was actually hybridised". Pure display arithmetic on numbers already computed
    by the sampling workers -- no new sampling."""
    if pct_value is None:
        return "--"
    nt_of = f"{mean_fraction * length:.1f}/{length} nt" if mean_fraction is not None else "--/-- nt"
    return f"{pct_value:.1%} ({nt_of}, n={n_samples})"


def _nt_n(fraction, length, n_samples):
    """Like ``_nt`` but with the sample count attached (fix #2, part (a)) -- used in
    the per-state detail rows where a bare nt-count was already shown but without
    saying how many Boltzmann draws it came from."""
    # 2026-10-05 fix (item K): same missing-denominator bug as `_nt` above.
    return "--" if fraction is None else f"{fraction * length:.1f}/{length} nt (n={n_samples})"


def _kofn_pct(k, n, mean_value):
    """stem_closed/misfolded/trigger_bound display (fix #5, root-bug fix 2026-10-05):
    shown as "k of N" using the SAME OPEN_THRESHOLD crossing already used for
    trigger_bound's "open" count (k = number of samples whose own per-sample fraction
    cleared the threshold), alongside the companion CONTINUOUS mean.

    ROOT BUG (2026-10-05): this used to return ``f"{_kofn(k, n)} ({_pct(mean_value)})"``
    -- "k/N (XX%)" where XX% was ``mean_value``, a continuous MEAN (e.g. the average
    per-sample fraction across all N draws) that is NOT k/n*100 and in general does not
    equal it (verified: "1018/1200 (87.7%)" when 1018/1200 = 84.8%; "16/1200 (29.3%)"
    when 16/1200 = 1.3%) -- two different statistics about the same measurement shown
    side by side as if they were one number. Fixed so the percentage next to k/N is
    ALWAYS literally k/n*100 (mathematically guaranteed to match), with the mean kept
    as a clearly, separately labelled second number -- "how often this crossed the
    open bar" vs. "how much of the span was hybridised on average", not one blurred
    into the other.
    """
    if k is None or n is None:
        return "--"
    freq_pct = k / n if n else 0.0
    return f"{_kofn(k, n)}<br><small>{freq_pct:.1%} of samples &middot; mean {_pct(mean_value)}</small>"

# ======================================================================================
# OFF-broadening (2026-10-05, user-approved -- parallels and_eu_report_spec.md's own
# Trap 5 reasoning: never silently narrow what counts as a leak to the one failure mode
# that is easiest to measure). OFF used to be picked by trigger-binding leak
# (frac_open / frac_both_open) alone, which is blind to a state whose STEM is
# independently loose for an unrelated reason while frac_open stays ~0 (nothing
# legitimate present to trigger-bind in the first place). Shared here so the snapshot-
# restore backfill (cell 9) and a fresh full run's own scoring cells (12, 19) score
# every candidate under the exact same formula -- never two near-identical copies that
# can quietly drift apart.
# ======================================================================================


def _single_badness(frac_open, n_stem_closed, n_samples):
    """badness = max(trigger-binding leak, stem-loosening leak) for one single-switch
    state. ``frac_open`` is the trigger-binding term (unchanged from before this fix).
    The stem-loosening term is ``1 - n_stem_closed / n_samples`` -- the FREQUENCY of
    samples where the stem failed to clear OPEN_THRESHOLD as "closed", reusing the
    exact already-stored ``n_stem_closed``/``n_samples`` integer counts (no new float
    rounding derivation, no resampling). Returns ``(badness, criterion_label, terms)``
    so the report can say which failure mode actually set OFF for a given candidate --
    2026-10-05 fix (single item 4 / AND item 1b): ``terms`` is the FULL dict every
    candidate term was computed from, not just the winner, so a reader can see both
    inputs to this ``max()`` call, not only which one won (``_off_breakdown_text``
    below renders it)."""
    terms = {"trigger-binding": frac_open}
    if n_stem_closed is not None and n_samples:
        terms["stem-loosening"] = 1.0 - (n_stem_closed / n_samples)
    criterion = max(terms, key=terms.get)
    return terms[criterion], criterion, terms


def _and_badness(frac_both_open, n_stem_closed_a, n_stem_closed_b, n_samples, *,
                  cross_talk_b=None, cross_talk_a=None):
    """AND counterpart of ``_single_badness``, extended with the two things the AND
    topology adds: per-side stem-loosening (A and B independently), and cross-talk --
    the OTHER hairpin opening without its own trigger present. ``cross_talk_b``/
    ``cross_talk_a`` are only ever passed by the caller for the states where they
    actually apply (an "A-side" state -- +A, +fakeA, +A+fakeA -- passes
    ``cross_talk_b=frac_b_open``; a "B-side" state passes ``cross_talk_a=frac_a_open``
    the same way; "alone" and "+A+B" pass neither -- A/B opening when their OWN
    trigger IS present, in +A+B, is the whole point, not a leak). Returns
    ``(badness, criterion_label, terms)``, same shape as ``_single_badness`` -- 2026-
    10-05 fix (AND item 1b): ``terms`` is the full dict, not just the winner."""
    terms = {"trigger-binding": frac_both_open}
    if n_stem_closed_a is not None and n_samples:
        terms["stem-loosening (A)"] = 1.0 - (n_stem_closed_a / n_samples)
    if n_stem_closed_b is not None and n_samples:
        terms["stem-loosening (B)"] = 1.0 - (n_stem_closed_b / n_samples)
    if cross_talk_b is not None:
        terms["cross-talk (B opens without its own trigger)"] = cross_talk_b
    if cross_talk_a is not None:
        terms["cross-talk (A opens without its own trigger)"] = cross_talk_a
    criterion = max(terms, key=terms.get)
    return terms[criterion], criterion, terms


def _and_off_term(label, state, mask_side=None):
    """Builds one ``(label, badness, mean_stem_closed_a, mean_stem_closed_b,
    criterion, terms)`` OFF candidate tuple for the AND scoring loop (cell 9's
    snapshot-restore path and cells 12/19's fresh-run path both call this, instead
    of each hand-rolling the masking rule below). ``terms`` (2026-10-05, AND item
    1b) is ``_and_badness``'s own full terms dict, passed through unchanged so the
    report can show every term considered for this state's own badness, not just
    the one that won.

    ``mask_side`` is which side's own stem-loosening term to SUPPRESS because that
    side's own REAL trigger is legitimately present in this state (``"a"`` for +A
    and +A+fakeA*, ``"b"`` for +B and +B+fakeB*, ``None`` for alone/+fakeA/+fakeB,
    where NEITHER real trigger is present and both sides' stem-loosening is a
    genuine signal). This is NOT optional detail: without it, a correctly-working
    design's own intended trigger binding -- which necessarily displaces that same
    side's stem_up from stem_down -- would score as "stem-loosening leak" on
    every single state that has any real trigger bound at all, collapsing OFF to
    ~100% uniformly across every candidate (verified directly while building this
    fix -- every AND candidate's ratio degenerated to ~0.0 before this mask was
    added). The masked side's cross-talk term (the OTHER side's frac_a_open/
    frac_b_open) still applies, same as before -- that side's own trigger is
    genuinely absent, so IT opening is still a real leak signal.
    """
    cross_talk_b = state.get("frac_b_open") if mask_side == "a" else None
    cross_talk_a = state.get("frac_a_open") if mask_side == "b" else None
    n_sc_a = None if mask_side == "a" else state.get("n_stem_closed_a")
    n_sc_b = None if mask_side == "b" else state.get("n_stem_closed_b")
    badness, criterion, terms = _and_badness(
        state["frac_both_open"], n_sc_a, n_sc_b, state.get("n_samples"),
        cross_talk_b=cross_talk_b, cross_talk_a=cross_talk_a,
    )
    return (label, badness, state["mean_stem_closed_a"], state["mean_stem_closed_b"],
            criterion, terms)


# ======================================================================================
# OFF breakdown display (2026-10-05, single item 4 / AND item 1b): a candidate's own
# card/row used to only name WHICH term won OFF's max() ("OFF set by: fakes' stem-
# loosening") -- never the actual number behind the term that lost, so a reader could
# not tell whether the losing term was a close second or nowhere close. This renders
# EVERY term _single_badness/_and_badness/_and_off_term actually considered, winner
# named first, from the exact same ``terms`` dict those functions now return -- pure
# display formatting on numbers already computed, no new measurement.
# ======================================================================================


def _off_breakdown_text(off_source, off_criterion, terms):
    """``"(OFF set by: {off_source}'s {off_criterion} -- {criterion} {pct} vs. ...)"``
    -- every term in ``terms`` (all in the same leak-fraction units as badness itself,
    high = bad), winner first, then every other term that was considered and lost."""
    if not terms:
        return f"(OFF set by: {html.escape(str(off_source))}'s {html.escape(str(off_criterion))})"
    ordered = [off_criterion] + [k for k in terms if k != off_criterion]
    parts = [f"{html.escape(k)} {_pct(terms[k])}" for k in ordered]
    return (f"(OFF set by: {html.escape(str(off_source))}'s {html.escape(str(off_criterion))} "
            f"&mdash; " + " vs. ".join(parts) + ")")


# Part 1 — single switch (no AND)

One gene, one trigger, one hairpin. Every state below is folded as a real multi-strand
complex through `FoldEngine.sample_structures` (never MFE-only, never a tube-style
proxy), and decomposed into `trigger_bound`/`stem_closed`/`misfolded` from the SAME
drawn structures (and_eu_report_spec.md Step 2). Output:
`var/reports/single_switch_report_<timestamp>.html/.pdf`.


In [ ]:
# ==================================================================================
# SNAPSHOT LOAD -- moved up here (right before Part 1 begins) from its earlier
# position at the end of the notebook: the single-switch and AND report cells
# (`single_report_rows`/`and_report_rows`) are both consumed WITHIN each Part,
# immediately after that Part's own heavy cells -- unlike the predecessor notebook,
# where every heavy cell of BOTH parts ran before any report cell, so restoring once
# right before the (single) block of report cells was enough. Restoring here, before
# anything else in either Part runs, means every "if SNAPSHOT_TO_LOAD is None:" heavy
# cell below (both Part 1 and Part 2) skips itself as designed, and both report
# cells find their rows already populated by the time they run -- same mechanism,
# just one restore point instead of two.
#
# `_report_root` (where every report/snapshot file in this notebook gets written) is
# also resolved HERE, once, for the same reason: both report-render cells need it,
# and a normal full run needs it before the (gated) snapshot-export cell can use it.
# ==================================================================================
_report_root = Path.cwd()
for _base in (_report_root, *_report_root.parents):
    if (_base / "pyproject.toml").exists():
        _report_root = _base / "var" / "reports"
        break
_report_root.mkdir(parents=True, exist_ok=True)

if SNAPSHOT_TO_LOAD is None:
    print("SNAPSHOT_TO_LOAD is None -- no-op (normal full run; nothing restored).")
else:
    _snap_path = Path(SNAPSHOT_TO_LOAD)
    if not _snap_path.is_absolute():
        _snap_path = _report_root / _snap_path
    if not _snap_path.exists():
        raise FileNotFoundError(f"{_snap_path} does not exist")
    _snap = json.loads(_snap_path.read_text())
    single_report_rows = _snap["single_report_rows"]
    and_report_rows = _snap["and_report_rows"]

    # JSON has no tuples: every (start, end) span came back as a 2-element list. The
    # report/plot code only ever indexes [0]/[1] or unpacks, which works the same for
    # a list -- but fuse()'s own callers and anything doing dict lookups keyed by a
    # span would not, so spans are put back to tuples rather than left half-converted
    # (same conversion the predecessor notebook's own LOAD SNAPSHOT cell applies).
    def _spans_to_tuples(obj):
        if isinstance(obj, dict):
            return {k: (tuple(v) if isinstance(v, list) and len(v) == 2
                        and all(isinstance(x, int) for x in v) else _spans_to_tuples(v))
                    for k, v in obj.items()}
        if isinstance(obj, list):
            return [_spans_to_tuples(v) for v in obj]
        return obj

    single_report_rows = _spans_to_tuples(single_report_rows)
    and_report_rows = _spans_to_tuples(and_report_rows)
    print(f"restored snapshot {_snap_path.name} (exported {_snap['exported_at']}): "
          f"{len(single_report_rows)} single-switch, {len(and_report_rows)} AND")

    # ------------------------------------------------------------------------------
    # BACKFILL (part 1 of 2): n_stem_closed/n_misfolded/n_unique_structures. This
    # snapshot was written BEFORE _single_decomp_worker/_and_decomp_worker carried
    # these fields, so they're backfilled here -- same "snapshot predates a new
    # field, backfill cheaply for just the report rows" pattern toehold_and_eu_test
    # .ipynb's own LOAD SNAPSHOT cell established for its dG-table backfill
    # (SingleDecompJob/AndDecompJob re-sampling here). A REAL re-sample through the
    # SAME updated worker functions -- not a placeholder -- so it won't reproduce the
    # snapshot's old means to the digit (sampling is stochastic), but it touches only
    # the states these ~40 kept rows actually need, never the full 300-candidate pool
    # or the 20x20 AND grid.
    #
    # MOVED AHEAD of the OFF-broadening rescoring below (2026-10-05): the new
    # badness formula needs n_stem_closed(_a/_b) to be present on every state dict,
    # so this backfill must run BEFORE that rescoring, not after it as in the
    # previous round (which only needed frac_open/mean_stem_closed, already present
    # on the old snapshot either way).
    # ------------------------------------------------------------------------------
    _backfill_jobs = []
    _backfill_slots = []  # (key, target dict to write results into)

    for _r in single_report_rows:
        _d = _r["design"]
        _fp = (_d["domains"]["toehold"][0], _d["domains"]["stem_up"][1])
        _su, _sd = _d["domains"]["stem_up"], _d["domains"]["stem_down"]
        _states = [("alone", "", _r["alone"]), ("real", _d["trigger"], _r["real"])]
        for _fk, _wd in enumerate(_r["fakes"]):
            _states.append((f"fake{_fk}", _wd["trigger"], _r["fake_results"][_fk]))
        for _state_name, _extra, _target in _states:
            if "n_stem_closed" in _target:
                continue
            _key = f"single|{_r['rank']}|{_state_name}"
            _backfill_jobs.append(SingleDecompJob(
                key=_key, switch=_d["switch"], extra=_extra, footprint=_fp,
                stem_up=_su, stem_down=_sd, temperature=TEMPERATURE_C,
                n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
            ))
            _backfill_slots.append((_key, _target))

    for _r in and_report_rows:
        _built = _r["built"]
        _fp5, _su5, _sd5 = _built["footprint_5p"], _built["domains_5p"]["stem_up"], _built["domains_5p"]["stem_down"]
        _fp3, _su3, _sd3 = _built["footprint_3p"], _built["domains_3p"]["stem_up"], _built["domains_3p"]["stem_down"]
        _ta, _tb = _r["d5"]["trigger"], _r["d3"]["trigger"]
        _and_states = [
            ("alone", (), _r["alone"]), ("+A", (_ta,), _r["state_a"]),
            ("+B", (_tb,), _r["state_b"]), ("+A+B", (_ta, _tb), _r["both"]),
        ]
        for _fk, _wd in enumerate(_r["fakes_a"]):
            _and_states.append((f"+fakeA{_fk}", (_wd["trigger"],), _r["fakeA_results"][_fk]))
            _and_states.append((f"+A+fakeA{_fk}", (_ta, _wd["trigger"]), _r["a_fakeA_results"][_fk]))
        for _fk, _wd in enumerate(_r["fakes_b"]):
            _and_states.append((f"+fakeB{_fk}", (_wd["trigger"],), _r["fakeB_results"][_fk]))
            _and_states.append((f"+B+fakeB{_fk}", (_tb, _wd["trigger"]), _r["b_fakeB_results"][_fk]))
        for _state_name, _extra, _target in _and_states:
            if "n_stem_closed_a" in _target:
                continue
            _key = f"and|{_r['key']}|{_state_name}"
            _backfill_jobs.append(AndDecompJob(
                key=_key, fused=_built["fused"], extra=_extra,
                footprint_a=_fp5, stem_up_a=_su5, stem_down_a=_sd5,
                footprint_b=_fp3, stem_up_b=_su3, stem_down_b=_sd3,
                temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
            ))
            _backfill_slots.append((_key, _target))

    if _backfill_jobs:
        _bt0 = time.time()
        _single_backfill_jobs = [j for j in _backfill_jobs if isinstance(j, SingleDecompJob)]
        _and_backfill_jobs = [j for j in _backfill_jobs if isinstance(j, AndDecompJob)]
        # One pool.map call per job TYPE (CLAUDE.md sec 5 / Trap 6) -- Single/And jobs
        # run through different worker functions, so they cannot share one call, but
        # every job of each type is batched into its own single call, never one
        # spin-up per job.
        _backfill_results = {}
        if _single_backfill_jobs:
            _backfill_results.update(run_parallel_single_decomp(_single_backfill_jobs, max_workers=MAX_WORKERS))
        if _and_backfill_jobs:
            _backfill_results.update(run_parallel_and_decomp(_and_backfill_jobs, max_workers=MAX_WORKERS))
        for _key, _target in _backfill_slots:
            _fresh = _backfill_results[_key]
            # The key's own prefix says which job type produced it ("single|.../"and|..."),
            # which is also exactly which field names apply -- no need to re-detect
            # from the target dict's shape.
            _fields = (
                ("n_stem_closed", "n_misfolded", "n_unique_structures")
                if _key.startswith("single|") else
                ("n_stem_closed_a", "n_misfolded_a", "n_stem_closed_b", "n_misfolded_b",
                 "n_a_open", "n_b_open", "n_both_open", "n_unique_structures")
            )
            for _field in _fields:
                _target[_field] = _fresh[_field]
        print(f"backfilled n_stem_closed/n_misfolded for {len(_backfill_jobs)} states "
              f"({len(_single_backfill_jobs)} single + {len(_and_backfill_jobs)} AND) in "
              f"{time.time() - _bt0:.1f}s, covering {len(single_report_rows)} single-switch "
              f"+ {len(and_report_rows)} AND report rows")
    else:
        print("snapshot already carries n_stem_closed/n_misfolded -- no backfill needed")

    # ------------------------------------------------------------------------------
    # RESCORING (part 2 of 2): OFF-broadening (2026-10-05, user-approved). Pure
    # arithmetic on numbers already sitting in the restored (and now backfilled)
    # per-state dicts -- no resampling, no FoldEngine call. Formula lives in cell 7
    # as `_single_badness`/`_and_badness`, shared with a fresh full run's own
    # scoring cells (12, 19) so every path scores identically.
    #
    # Previously OFF was picked by trigger-binding leak (frac_open/frac_both_open)
    # alone -- blind to a state whose STEM is independently loose for an unrelated
    # reason while frac_open stays ~0 (nothing legitimate present to trigger-bind).
    # badness = max(trigger-binding leak, either hairpin's own stem failing to stay
    # closed[, for AND: the OTHER hairpin opening without its own trigger present]).
    # `off_criterion` records exactly which term won, per candidate, so the report
    # can say e.g. "(OFF set by: fakes' stem-loosening)" instead of leaving FAKES and
    # OFF silently about two different, unstated things.
    # ------------------------------------------------------------------------------
    for _r in single_report_rows:
        _alone, _real, _fake_results = _r["alone"], _r["real"], _r["fake_results"]
        _badness_alone, _crit_alone, _terms_alone = _single_badness(
            _alone["frac_open"], _alone.get("n_stem_closed"), _alone.get("n_samples"))
        _off_candidates = [("alone", _badness_alone, _alone["mean_stem_closed"], _crit_alone, _terms_alone)]
        _fake_mean_stem_closed = None
        if _fake_results:
            _mean_fake_open = sum(x["frac_open"] for x in _fake_results) / len(_fake_results)
            _fake_mean_stem_closed = sum(x["mean_stem_closed"] for x in _fake_results) / len(_fake_results)
            _sum_fake_n_sc = sum(x["n_stem_closed"] for x in _fake_results)
            _sum_fake_n = sum(x["n_samples"] for x in _fake_results)
            _badness_fakes, _crit_fakes, _terms_fakes = _single_badness(_mean_fake_open, _sum_fake_n_sc, _sum_fake_n)
            _off_candidates.append(("fakes", _badness_fakes, _fake_mean_stem_closed, _crit_fakes, _terms_fakes))
        _off_source, _off, _off_mean_stem_closed, _off_criterion, _off_terms = max(
            _off_candidates, key=lambda t: t[1])
        _on = _real["frac_open"]
        _r["on"], _r["off"] = _on, _off
        _r["off_source"] = _off_source
        _r["off_criterion"] = _off_criterion
        _r["off_terms"] = _off_terms
        _r["off_mean_stem_closed"] = _off_mean_stem_closed
        _r["fake_mean_stem_closed"] = _fake_mean_stem_closed
        _r["ratio"] = (_on * 100.0) / (1.0 + _off * 100.0)
        _r["on_minus_off"] = _on - _off
        _r.pop("is_bound", None)
    single_report_rows.sort(key=lambda r: -r["ratio"])

    for _r in and_report_rows:
        _alone, _a, _b, _both = _r["alone"], _r["state_a"], _r["state_b"], _r["both"]
        _fakeA_results, _fakeB_results = _r["fakeA_results"], _r["fakeB_results"]
        _a_fakeA_results, _b_fakeB_results = _r["a_fakeA_results"], _r["b_fakeB_results"]

        def _mean_field(results, field):
            return sum(x[field] for x in results) / len(results) if results else None

        def _sum_field(results, field):
            return sum(x[field] for x in results) if results else None

        def _pooled_state(results):
            # Pooled synthetic "state" dict over several fake draws -- summed exact
            # counts / summed n_samples for a real pooled k/N, mean of the
            # continuous means -- same shape _and_off_term expects from a real state.
            return {
                "frac_both_open": _mean_field(results, "frac_both_open"),
                "frac_a_open": _mean_field(results, "frac_a_open"),
                "frac_b_open": _mean_field(results, "frac_b_open"),
                "n_stem_closed_a": _sum_field(results, "n_stem_closed_a"),
                "n_stem_closed_b": _sum_field(results, "n_stem_closed_b"),
                "n_samples": _sum_field(results, "n_samples"),
                "mean_stem_closed_a": _mean_field(results, "mean_stem_closed_a"),
                "mean_stem_closed_b": _mean_field(results, "mean_stem_closed_b"),
            }

        # mask_side: which side's own stem-loosening is SUPPRESED in a given state
        # because that side's own REAL trigger is legitimately present there (see
        # _and_off_term's own docstring, cell 7) -- "alone"/"+fakeA"/"+fakeB" mask
        # neither side (no real trigger present at all, so both sides' own
        # stem-loosening is a genuine signal); "+A"/"+A+fakeA" mask "a"; "+B"/
        # "+B+fakeB" mask "b".
        _off_candidates = [
            _and_off_term("alone", _alone, mask_side=None),
            _and_off_term("+A", _a, mask_side="a"),
            _and_off_term("+B", _b, mask_side="b"),
        ]
        for _label, _results, _mask in (
            ("+fakeA", _fakeA_results, None), ("+fakeB", _fakeB_results, None),
            ("+A+fakeA", _a_fakeA_results, "a"), ("+B+fakeB", _b_fakeB_results, "b"),
        ):
            if not _results:
                continue
            _off_candidates.append(_and_off_term(_label, _pooled_state(_results), mask_side=_mask))
        _off_source, _off, _off_mean_sc_a, _off_mean_sc_b, _off_criterion, _off_terms = max(
            _off_candidates, key=lambda t: t[1])

        _on = _both["frac_both_open"]
        _r["on"], _r["off"] = _on, _off
        _r["off_source"] = _off_source
        _r["off_criterion"] = _off_criterion
        _r["off_terms"] = _off_terms
        _r["off_mean_stem_closed_a"] = _off_mean_sc_a
        _r["off_mean_stem_closed_b"] = _off_mean_sc_b
        _r["fakeA_mean_stem_closed"] = _mean_field(_fakeA_results, "mean_stem_closed_a")
        _r["fakeB_mean_stem_closed"] = _mean_field(_fakeB_results, "mean_stem_closed_b")
        _r["ratio"] = (_on * 100.0) / (1.0 + _off * 100.0)
        _r["on_minus_off"] = _on - _off
        _r.pop("is_bound", None)
    and_report_rows.sort(key=lambda r: -r["ratio"])
    print(f"re-scored {len(single_report_rows)} single-switch + {len(and_report_rows)} AND "
          f"rows under the OFF-broadened badness formula (max of trigger-binding leak, "
          f"stem-loosening[, AND cross-talk]) -- pure arithmetic, no resampling; both "
          f"lists re-sorted by the new ratio.")


In [6]:
if SNAPSHOT_TO_LOAD is None:
    # FILTER_HOMOPOLYMER_RUN: load the WHOLE file, filter, THEN cut to SINGLE_POOL_N --
    # filtering a pre-capped pool would silently shrink it instead of backfilling with
    # the next-best candidates that pass (and_eu_report_spec.md Step 3.1).
    _load_limit = None if FILTER_HOMOPOLYMER_RUN else SINGLE_POOL_N
    single_designs, single_problems = load_ranked_designs(
        CSV_PATH, folder, columns=COLUMNS, carry_metrics=CARRY_METRICS, limit=_load_limit,
    )
    for _line_no, _reason in single_problems[:10]:
        print(f"  skipped line {_line_no}: {_reason}")

    if FILTER_HOMOPOLYMER_RUN:
        _n_before = len(single_designs)
        single_designs = [
            d for d in single_designs
            if "toehold" not in d["domains"]
            or longest_homopolymer(
                d["switch"][d["domains"]["toehold"][0]:d["domains"]["toehold"][1]]
            )[1] <= MAX_HOMOPOLYMER_RUN
        ]
        print(f"  FILTER_HOMOPOLYMER_RUN: {_n_before} loaded -> {len(single_designs)} passed "
              f"-> top {min(len(single_designs), SINGLE_POOL_N)} of those kept")
        single_designs = single_designs[:SINGLE_POOL_N]

    print(f"loaded {len(single_designs)} designs from {CSV_PATH} "
          f"({len(single_problems)} rows skipped)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the single-switch design load.")


SNAPSHOT_TO_LOAD is set -- skipping the single-switch design load.


In [7]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # SINGLE-SWITCH PARALLEL SCREEN. For every candidate: alone, +real trigger, and
    # +each of up to SINGLE_N_FAKES distinct fakes -- all states of all candidates
    # submitted as ONE list to ONE run_parallel_single_decomp call (CLAUDE.md sec 5 /
    # Trap 6: one process-pool call per batch, never one spin-up per job).
    #
    # Fake selection (Trap 2 / Trap 3): pick_fake_design screens by SEQUENCE
    # complementarity (never transcript position), with wobble tolerance and an
    # absolute cap (FAKE_MAX_SEQ_OVERLAP_NT, WOBBLE_MAX_FRACTION) -- not every
    # candidate finds SINGLE_N_FAKES qualifying fakes; whatever count IS found is used
    # and reported, never padded or silently treated as a full set.
    # ==============================================================================
    _t0 = time.time()
    _single_jobs = []
    _single_fakes_by_rank = {}

    for _i, _d in enumerate(single_designs):
        _fp = (_d["domains"]["toehold"][0], _d["domains"]["stem_up"][1])
        _su, _sd = _d["domains"]["stem_up"], _d["domains"]["stem_down"]
        _single_jobs.append(SingleDecompJob(
            key=f"{_d['rank']}|alone", switch=_d["switch"], extra="",
            footprint=_fp, stem_up=_su, stem_down=_sd,
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
        ))
        _single_jobs.append(SingleDecompJob(
            key=f"{_d['rank']}|real", switch=_d["switch"], extra=_d["trigger"],
            footprint=_fp, stem_up=_su, stem_down=_sd,
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
        ))

        _picked = {_d["rank"]}
        _fakes = []
        for _k in range(SINGLE_N_FAKES):
            _wd = pick_fake_design(
                _d, _d, _picked, single_designs, start_offset=_i + 1 + _k,
                max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT, wobble_max_fraction=WOBBLE_MAX_FRACTION,
            )
            if _wd is None:
                break  # pool exhausted -- fewer than SINGLE_N_FAKES qualifying fakes exist
            _picked.add(_wd["rank"])
            _fakes.append(_wd)
        _single_fakes_by_rank[_d["rank"]] = _fakes
        for _fk, _wd in enumerate(_fakes):
            _single_jobs.append(SingleDecompJob(
                key=f"{_d['rank']}|fake{_fk}", switch=_d["switch"], extra=_wd["trigger"],
                footprint=_fp, stem_up=_su, stem_down=_sd,
                temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
            ))

    _n_short = sum(1 for _f in _single_fakes_by_rank.values() if len(_f) < SINGLE_N_FAKES)
    print(f"{len(single_designs)} candidates, {len(_single_jobs)} jobs "
          f"(2 base + up to {SINGLE_N_FAKES} fakes each); "
          f"{_n_short} candidates found fewer than {SINGLE_N_FAKES} qualifying fakes")

    single_decomp_by_key = run_parallel_single_decomp(_single_jobs, max_workers=MAX_WORKERS)
    _elapsed = time.time() - _t0
    print(f"done in {_elapsed:.0f}s ({_elapsed / len(_single_jobs) * 1000:.1f} ms/job)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the single-switch parallel screen.")


SNAPSHOT_TO_LOAD is set -- skipping the single-switch parallel screen.


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ON/OFF RATIO (and_eu_report_spec.md Step 2 / Trap 5):
    #   ON  = trigger_bound(+real trigger) -- here, the frac_open FREQUENCY of the
    #         +real-trigger state.
    #   OFF = max(badness over: alone, mean over SINGLE_N_FAKES fakes) -- the worst
    #         across genuinely illegitimate STATES, never "1 - ON" (Trap 5). badness
    #         (2026-10-05, OFF-broadening) is itself max(trigger-binding leak,
    #         stem-loosening leak) per state -- `_single_badness` in cell 7, shared
    #         with the snapshot-restore path (cell 9) so a fresh run and a restored
    #         run score identically.
    #
    # ratio = on_pct / (1 + off_pct), with on_pct = on*100, off_pct = off*100 --
    # PERCENTAGE POINTS, not 0-1 fractions (2026-10-05 feedback fix #3, replacing the
    # old floor-at-1/N_SAMPLES + ">=N" bound notation entirely). Why percentage
    # points: at fraction units the "+1" in the denominator swamps any real OFF
    # signal below 100% -- OFF=0.5% (0.005) and OFF=2% (0.02) would give denominators
    # of 1.005 vs 1.02, barely different, even though a design leaking 4x more than
    # another is a very different design. In percentage-point units (0.5 vs 2.0) the
    # same two OFF values give denominators of 1.5 vs 3.0 -- a real, visible
    # separation -- while OFF=0 still gives a clean, finite, ALWAYS-computable ratio
    # (on_pct / 1) with no special-cased "bound" to explain or display.
    # ==============================================================================
    single_scored = []
    for _d in single_designs:
        _rank = _d["rank"]
        _alone = single_decomp_by_key[f"{_rank}|alone"]
        _real = single_decomp_by_key[f"{_rank}|real"]
        _fakes = _single_fakes_by_rank[_rank]
        _fake_results = [single_decomp_by_key[f"{_rank}|fake{_fk}"] for _fk in range(len(_fakes))]

        _on = _real["frac_open"]
        # Track which ILLEGITIMATE state actually sets OFF (and that state's own
        # mean_stem_closed, and now which CRITERION -- trigger-binding vs
        # stem-loosening -- won inside that state) alongside the max itself.
        _badness_alone, _crit_alone, _terms_alone = _single_badness(
            _alone["frac_open"], _alone.get("n_stem_closed"), _alone.get("n_samples"))
        _off_candidates = [("alone", _badness_alone, _alone["mean_stem_closed"], _crit_alone, _terms_alone)]
        _fake_mean_stem_closed = None
        if _fake_results:
            _mean_fake_open = sum(r["frac_open"] for r in _fake_results) / len(_fake_results)
            _fake_mean_stem_closed = sum(r["mean_stem_closed"] for r in _fake_results) / len(_fake_results)
            _sum_fake_n_sc = sum(r["n_stem_closed"] for r in _fake_results)
            _sum_fake_n = sum(r["n_samples"] for r in _fake_results)
            _badness_fakes, _crit_fakes, _terms_fakes = _single_badness(_mean_fake_open, _sum_fake_n_sc, _sum_fake_n)
            _off_candidates.append(("fakes", _badness_fakes, _fake_mean_stem_closed, _crit_fakes, _terms_fakes))
        _off_source, _off, _off_mean_stem_closed, _off_criterion, _off_terms = max(
            _off_candidates, key=lambda t: t[1])

        _on_pct, _off_pct = _on * 100.0, _off * 100.0
        _ratio = _on_pct / (1.0 + _off_pct)

        single_scored.append({
            "design": _d, "rank": _rank,
            "alone": _alone, "real": _real, "fakes": _fakes, "fake_results": _fake_results,
            "on": _on, "off": _off, "off_source": _off_source, "off_criterion": _off_criterion,
            "off_terms": _off_terms,
            "off_mean_stem_closed": _off_mean_stem_closed,
            "fake_mean_stem_closed": _fake_mean_stem_closed,
            "ratio": _ratio, "on_minus_off": _on - _off,
        })

    single_scored.sort(key=lambda r: -r["ratio"])
    single_report_rows = single_scored[:SINGLE_REPORT_N]
    print(f"{len(single_scored)} candidates scored; top {len(single_report_rows)} "
          f"(ranks {[r['rank'] for r in single_report_rows]}) go to the report")
    print(f"  best ratio {single_scored[0]['ratio']:.1f}, "
          f"on={single_scored[0]['on']:.1%}, off={single_scored[0]['off']:.1%} "
          f"(off_source={single_scored[0]['off_source']}, off_criterion={single_scored[0]['off_criterion']})")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping single-switch scoring.")


In [9]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ENERGIES (ensemble free energy via FoldEngine.partition, kcal/mol, more negative
    # = more stable) -- computed ONLY for the top SINGLE_REPORT_N candidates that
    # reach the report; these are plain cached scalar lookups (not sampling), so
    # serial calls are fine here (CLAUDE.md sec 5 / sec 6: one FoldEngine, same cache).
    # and_eu_report_spec.md Step 2: "switch alone, trigger alone, switch+trigger, dG,
    # and the mean dG across the SINGLE_N_FAKES fakes".
    # ==============================================================================
    for _r in single_report_rows:
        _d = _r["design"]
        _switch, _trigger = _d["switch"], _d["trigger"]
        _ee_switch = folder.partition(_switch)
        _ee_trigger = folder.partition(_trigger)
        _ee_on = folder.partition(f"{_switch}&{_trigger}")
        _r["ee_switch"] = _ee_switch
        _r["ee_trigger"] = _ee_trigger
        _r["ee_switch_trigger"] = _ee_on
        _r["d_ee"] = _ee_on - _ee_switch
        _fake_d_ees = [
            folder.partition(f"{_switch}&{_wd['trigger']}") - _ee_switch
            for _wd in _r["fakes"]
        ]
        _r["mean_d_ee_fakes"] = sum(_fake_d_ees) / len(_fake_d_ees) if _fake_d_ees else None
    print(f"energies computed for {len(single_report_rows)} report candidates")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping single-switch energies.")


SNAPSHOT_TO_LOAD is set -- skipping single-switch energies.


In [10]:
# Images -- UNGATED (always runs): rebuilds from whichever rows exist, fresh or
# restored from a snapshot (snapshot never stores images -- predecessor notebook's
# own established rationale: base64 PNGs would bloat every snapshot for nothing a
# fresh render in seconds does not already give back).
# _PAYLOAD_DOMAINS now lives in cell 7 (shared with the AND images cell below --
# 2026-10-05 fix, single item 11 / AND item 12: one payload-domain list, not two
# copies that could silently drift apart).

for _r in single_report_rows:
    _d = _r["design"]
    _switch = _d["switch"]
    _domains = _d["domains"]
    _toehold_span = _domains["toehold"]
    _stem_up_span = _domains["stem_up"]
    _stem_down_span = _domains["stem_down"]
    _footprint = (_toehold_span[0], _stem_up_span[1])

    # 2D structure + bpp matrix, OFF (alone) and ON (+real trigger) -- through Kozak
    # only (payload dropped), a real separate fold of the shorter sequence so no
    # dangling bonds into a cropped-out tail (same convention as the predecessor).
    _kept = {n: s for n, s in _domains.items() if n not in _PAYLOAD_DOMAINS}
    _display_end = max((s[1] for s in _kept.values()), default=len(_switch))
    _display_seq = _switch[:_display_end]
    # Real per-domain fill color (fix #6), not the old flat fill -- every base in a
    # known domain span gets that domain's own color; a base outside every span (none
    # here, since _kept covers the whole display sequence) keeps the default fill.
    _domain_fill = _domain_fill_from_domains(_kept, _DOMAIN_COLOR_A)

    _mfe_off = folder.mfe(_display_seq)
    _matrix_off = folder.base_pair_probabilities(_display_seq)
    _r["structure_off_png"] = structure_png(
        _display_seq, _mfe_off.structure, f"rank {_r['rank']} — OFF (alone, through Kozak)",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)], domain_fill=_domain_fill,
    )
    # No second strand in the OFF fold -- no junction to mark, so no boundary crosshair
    # (fix #7: a boundary is only drawn for a matrix that actually includes a trigger).
    _r["matrix_off_png"] = bpp_heatmap_png(
        _display_seq, _matrix_off, "OFF: base-pair probabilities",
    )

    _bound_strands = f"{_display_seq}&{_d['trigger']}"
    _bound_concat = _display_seq + _d["trigger"]  # structure/bpp index this, "&" removed
    # 2026-10-05 fix (item 10): the trigger strand used to have no domain_fill entry
    # at all, so it rendered in flat _DEFAULT_FILL on this plot -- give it its own
    # distinct color (_TRIGGER_COLOR), scoped to this ON-state image only (the OFF
    # plot above has no trigger strand present).
    _domain_fill_on = [*_domain_fill, (len(_display_seq), len(_bound_concat), _TRIGGER_COLOR, "trigger")]
    _mfe_on = folder.mfe(_bound_strands)
    _matrix_on = folder.base_pair_probabilities(_bound_strands)
    _r["structure_on_png"] = structure_png(
        _bound_concat, _mfe_on.structure, f"rank {_r['rank']} — ON (+real trigger)",
        spans=[(_toehold_span[0], _toehold_span[1], _COLOR_A)], domain_fill=_domain_fill_on,
    )
    # One real strand junction present (switch -> trigger) at x=y=len(switch) -- the
    # single dashed crosshair fix #7 asks for, in the trigger's own color.
    _r["matrix_on_png"] = bpp_heatmap_png(
        _bound_concat, _matrix_on, "ON: base-pair probabilities",
        boundaries=[(len(_display_seq), _COLOR_A)],
    )

    # Domain diagram: layout bar (image) + each domain's own sequence. Built from
    # _kept (exp_gene/opt_exp_gene already excluded above), not the full _domains --
    # 2026-10-05 fix (item 11): the layout bar used to draw ALL domains including the
    # payload, inconsistent with the structure/matrix plots above which already only
    # ever saw _display_seq (truncated before the payload starts). The payload domain
    # itself is untouched in the underlying design dict -- only what gets drawn here
    # changes.
    _segments = [
        (s, name, e - s, _DOMAIN_COLOR_A.get(name, "#B7C4BC"), _switch[s:e])
        for name, (s, e) in sorted(_kept.items(), key=lambda kv: kv[1][0])
        if e > s
    ]
    # Total switch length shown on the bar itself (item 11) -- the full fused
    # sequence length, excluding the trigger (which is never part of `_switch`).
    _r["layout_png"] = layout_bar_png(
        _segments, len(_switch),
        f"rank {_r['rank']} — domain layout (switch length {len(_switch)} nt)",
    )
    _r["domain_rows"] = [(name, length, seq) for (_s, name, length, _c, seq) in _segments]

print(f"images built for {len(single_report_rows)} single-switch candidates")


images built for 20 single-switch candidates


In [ ]:
# ==================================================================================
# SINGLE-SWITCH REPORT -- HTML (+ PDF). One candidate card per report row: energies,
# the three-way state decomposition per state, Boltzmann agreement as "k of N", nt of
# trigger hybridised / nt of stem still hybridised, probability matrices, domain
# diagram with per-domain sequence. Every metric gets a one-line explanation
# (and_eu_report_spec.md Step 4 -- "written so someone new to the project can read it").
# ==================================================================================

def render_single_report_html(rows):
    summary_trs = []
    for i, r in enumerate(rows, start=1):
        _d = r["design"]
        _fp_len = _d["domains"]["stem_up"][1] - _d["domains"]["toehold"][0]
        _su_len = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
        _off_n_samples = r["alone"]["n_samples"] if r["off_source"] == "alone" else N_SAMPLES
        # OFF (fakes) column -- fix #4: replaces the old bare "{len(fakes)}/{N}" count
        # with the actual OFF nt-count metric for the with-fake check specifically:
        # mean nt of stem_up still bound to stem_down across the fake draws. Shown
        # with its own mean % (fix #1, 2026-10-05) via _nt_pct, not a bare nt-count.
        _fake_off_nt = _nt_pct(r["fake_mean_stem_closed"], _su_len)
        summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{html.escape(str(r['rank']))}</td>"
            f'<td class="num">{_pct_nt_n(r["on"], r["real"]["mean_trigger_bound"], _fp_len, r["real"]["n_samples"])}</td>'
            f'<td class="num">{_pct_nt_n(r["off"], r["off_mean_stem_closed"], _su_len, _off_n_samples)}</td>'
            f'<td class="num">{r["ratio"]:.1f}</td>'
            f'<td class="num">{_pct(r["on_minus_off"])}</td>'
            f'<td class="num">{_num(r["d_ee"])}</td>'
            f'<td class="num">{_fake_off_nt}</td>'
            f'<td><a href="#single-{i}">detail &darr;</a></td>'
            "</tr>"
        )

    cards = []
    for i, r in enumerate(rows, start=1):
        _d = r["design"]
        _fp_len = _d["domains"]["stem_up"][1] - _d["domains"]["toehold"][0]
        _su_len = _d["domains"]["stem_up"][1] - _d["domains"]["stem_up"][0]
        _alone, _real = r["alone"], r["real"]
        _fake_results = r["fake_results"]
        _mean_fake_tb = (sum(x["mean_trigger_bound"] for x in _fake_results) / len(_fake_results)
                         if _fake_results else None)
        _mean_fake_sc = (sum(x["mean_stem_closed"] for x in _fake_results) / len(_fake_results)
                         if _fake_results else None)
        _mean_fake_mf = (sum(x["mean_misfolded"] for x in _fake_results) / len(_fake_results)
                         if _fake_results else None)
        _mean_fake_open = (sum(x["frac_open"] for x in _fake_results) / len(_fake_results)
                           if _fake_results else None)
        # k-of-N for stem_closed/misfolded (fix #5) over the fake draws: no single
        # "n_stem_closed" count exists across several averaged states, so this sums
        # the per-draw counts and the per-draw sample sizes -- still a real k of N,
        # just pooled across the SINGLE_N_FAKES draws rather than one state's own.
        _sum_fake_n_sc = sum(x["n_stem_closed"] for x in _fake_results) if _fake_results else None
        _sum_fake_n_mf = sum(x["n_misfolded"] for x in _fake_results) if _fake_results else None
        _sum_fake_n = sum(x["n_samples"] for x in _fake_results) if _fake_results else None
        # Mean unique-structures count across the pooled fake draws (item N) -- a
        # mean of counts, not a sum, since each draw's own uniqueness is relative to
        # ITS OWN N_SAMPLES draws, not comparable across draws the way the
        # threshold-crossing counts above are.
        _mean_fake_unique = (sum(x["n_unique_structures"] for x in _fake_results) / len(_fake_results)
                             if _fake_results else None)

        decomp_rows = f"""
          <tr><td><b>alone</b> (no trigger)</td>
              <td class="num">{_pct(_alone["mean_trigger_bound"])}</td>
              <td class="num">{_kofn_pct(_alone["n_stem_closed"], _alone["n_samples"], _alone["mean_stem_closed"])}</td>
              <td class="num">{_kofn_pct(_alone["n_misfolded"], _alone["n_samples"], _alone["mean_misfolded"])}</td>
              <td class="num">{_kofn(_alone["n_open"], _alone["n_samples"])}</td>
              <td class="num">{_nt(_alone["mean_stem_closed"], _su_len)}</td>
              <td class="num">{_kofn(_alone["n_unique_structures"], _alone["n_samples"])}</td></tr>
          <tr><td><b>+real trigger</b> (ON)</td>
              <td class="num">{_pct(_real["mean_trigger_bound"])}</td>
              <td class="num">{_kofn_pct(_real["n_stem_closed"], _real["n_samples"], _real["mean_stem_closed"])}</td>
              <td class="num">{_kofn_pct(_real["n_misfolded"], _real["n_samples"], _real["mean_misfolded"])}</td>
              <td class="num">{_kofn(_real["n_open"], _real["n_samples"])}</td>
              <td class="num">{_nt(_real["mean_trigger_bound"], _fp_len)}</td>
              <td class="num">{_kofn(_real["n_unique_structures"], _real["n_samples"])}</td></tr>
        """
        if _fake_results:
            decomp_rows += f"""
              <tr><td>+fake trigger (mean of {len(_fake_results)}/{SINGLE_N_FAKES})</td>
                  <td class="num">{_pct(_mean_fake_tb)}</td>
                  <td class="num">{_kofn_pct(_sum_fake_n_sc, _sum_fake_n, _mean_fake_sc)}</td>
                  <td class="num">{_kofn_pct(_sum_fake_n_mf, _sum_fake_n, _mean_fake_mf)}</td>
                  <td class="num">{_pct(_mean_fake_open)} open</td>
                  <td class="num">{_nt_pct(_mean_fake_tb, _fp_len)}</td>
                  <td class="num">mean {_mean_fake_unique:.0f}/{N_SAMPLES}</td></tr>
            """
        else:
            # 2026-10-05 fix (item 3 / item 6): a candidate whose trigger already has
            # an unusually SHORT complementary run against its own footprint (checked
            # directly -- this report's own #1 candidate by ratio has a real run of
            # only 4nt) means almost nothing in the pool can score even lower under
            # pick_fake_design's self-calibrated "beat the real trigger" screen, so
            # 0 qualifying fakes is a legitimate screening-availability outcome, not a
            # bug -- but it must say so, not silently render as a row of "--".
            decomp_rows += f"""
              <tr><td>+fake trigger</td>
                  <td class="scr-cell-note" colspan="6"><i>no qualifying fakes found (0/{SINGLE_N_FAKES}
                  available): no pool candidate had a complementarity run against this
                  footprint below this trigger's own (&le;{FAKE_MAX_SEQ_OVERLAP_NT} nt
                  overlap cap, &le;{WOBBLE_MAX_FRACTION:.0%} wobble) -- this candidate's
                  real trigger is already tightly specific, so no fake-based OFF check
                  ran for it.</i></td></tr>
            """

        energy_rows = f"""
          <tr><td>switch alone</td><td class="num">{_num(r["ee_switch"])}</td></tr>
          <tr><td>trigger alone</td><td class="num">{_num(r["ee_trigger"])}</td></tr>
          <tr><td>switch + trigger</td><td class="num">{_num(r["ee_switch_trigger"])}</td></tr>
          <tr><td><b>dG (switch+trigger - switch alone)</b></td><td class="num">{_num(r["d_ee"])}</td></tr>
          <tr><td>mean dG across {len(r["fakes"])} fakes</td><td class="num">{_num(r["mean_d_ee_fakes"])}</td></tr>
        """

        domain_block = "".join(
            f'<div class="scr-seq-block"><span class="label">{html.escape(name)} ({length} nt)</span>'
            f'<span class="scr-mono">{html.escape(seq)}</span></div>'
            for name, length, seq in r["domain_rows"]
        )

        # 2026-10-05 fix (item 2, upgraded this round -- single item 4): which
        # criterion -- trigger-binding vs stem-loosening -- actually determined
        # off_source for THIS candidate, so the FAKES column and the OFF column are
        # never silently about two different, unstated things again. Now shows BOTH
        # terms' own numbers (`_off_breakdown_text`, cell 7), not just the winning
        # term's name -- a reader can see both inputs to the max(), not only which
        # one won.
        _off_note = _off_breakdown_text(r["off_source"], r["off_criterion"], r.get("off_terms"))

        cards.append(f"""
        <div class="scr-candidate" id="single-{i}">
          <div class="scr-candidate-head">
            <h3>#{i} — rank {html.escape(str(r["rank"]))}</h3>
            <span class="scr-candidate-id">ON {_pct_nt_n(r["on"], r["real"]["mean_trigger_bound"], _fp_len, r["real"]["n_samples"])} &middot; OFF {_pct_nt_n(r["off"], r["off_mean_stem_closed"], _su_len, r["alone"]["n_samples"] if r["off_source"] == "alone" else N_SAMPLES)} {html.escape(_off_note)} &middot; ratio {r["ratio"]:.1f}</span>
          </div>
          <p class="scr-p"><b>Trigger ({len(_d["trigger"])} nt):</b> <span class="scr-mono">{html.escape(_d["trigger"])}</span></p>
          <p class="scr-p">Energies are ensemble &Delta;G in kcal/mol; more negative = more stable.</p>
          <div class="scr-table-wrap scr-table-energy" style="max-width:420px;"><table>
            <thead><tr><th>energy</th><th class="num">kcal/mol</th></tr></thead>
            <tbody>{energy_rows}</tbody>
          </table></div>
          <div class="scr-table-wrap"><table style="table-layout:fixed; width:100%;">
            <colgroup>
              <col style="width:16%"><col style="width:14%"><col style="width:16%">
              <col style="width:16%"><col style="width:12%"><col style="width:13%"><col style="width:13%">
            </colgroup>
            <thead><tr><th>state</th><th class="num">trigger_bound<br><small>ON signal</small></th>
              <th class="num">stem_closed<br><small>k/N</small></th>
              <th class="num">misfolded<br><small>k/N</small></th>
              <th class="num">open<br><small>k/N</small></th>
              <th class="num">nt hybridised</th>
              <th class="num">unique<br><small>structures</small></th></tr></thead>
            <tbody>{decomp_rows}</tbody>
          </table></div>
          <img class="scr-img" src="data:image/png;base64,{r["layout_png"]}">
          <div class="scr-seq-block">{domain_block}</div>
          <div class="scr-structure-grid">
            <figure><img src="data:image/png;base64,{r["structure_off_png"]}"><figcaption>OFF structure</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["structure_on_png"]}">
              <figcaption>ON structure (trigger drawn after the switch for layout only --
                FoldEngine's "&amp;"-joined dimer convention; only the switch's OWN
                internal layout here is meaningful, not where the trigger is drawn)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_off_png"]}"><figcaption>OFF probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_on_png"]}"><figcaption>ON probability matrix (dashed crosshair = switch/trigger boundary)</figcaption></figure>
          </div>
        </div>
        """)

    # Dynamic worked example for the glossary (fix #4, 2026-10-05) -- real numbers
    # from THIS render's own #1 candidate, never a stale hand-typed number.
    _ex = rows[0]
    _ex_alone = _ex["alone"]
    _ex_k, _ex_n_s, _ex_mean = (_ex_alone["n_misfolded"], _ex_alone["n_samples"],
                                 _ex_alone["mean_misfolded"])
    _ex_freq = _ex_k / _ex_n_s if _ex_n_s else 0.0
    _worked_example = (
        f"e.g. for this report's own #1 candidate (rank {html.escape(str(_ex['rank']))}), "
        f"alone state: <code>misfolded</code> = {_kofn(_ex_k, _ex_n_s)} ({_ex_freq:.1%} of "
        f"samples &middot; mean {_pct(_ex_mean)}) means -- in {_ex_k} of the {_ex_n_s} "
        f"Boltzmann draws did the misfolded fraction individually reach the "
        f"{OPEN_THRESHOLD:.0%} OPEN_THRESHOLD in that single sample, but averaged across "
        f"all {_ex_n_s} draws, {_pct(_ex_mean)} of the FOOTPRINT's nt were, on average, "
        f"paired to something other than their own stem_down."
    )

    # Second worked example (2026-10-05, single item 2): stem_closed's own k/N-vs-mean
    # split, specifically -- picked as the FIRST real state across every report row's
    # own alone/fake draws whose k/N frequency and continuous mean diverge by a real
    # margin (>=3 points), so the worked numbers actually illustrate the distinction
    # rather than landing on a state where the two happen to coincide. Real numbers
    # from THIS render, never hand-typed.
    _stem_example = None
    for _rr in rows:
        _su_len_ex = (_rr["design"]["domains"]["stem_up"][1]
                      - _rr["design"]["domains"]["stem_up"][0])
        for _label_ex, _st in [("alone", _rr["alone"])] + [("fake", x) for x in _rr["fake_results"]]:
            _k_ex, _n_ex, _m_ex = (_st.get("n_stem_closed"), _st.get("n_samples"),
                                    _st.get("mean_stem_closed"))
            if _k_ex is None or not _n_ex or _m_ex is None:
                continue
            _freq_ex = _k_ex / _n_ex
            if abs(_freq_ex - _m_ex) >= 0.03:
                _stem_example = (_rr, _label_ex, _k_ex, _n_ex, _m_ex, _su_len_ex, _freq_ex)
                break
        if _stem_example:
            break
    if _stem_example:
        _ex_rr2, _ex_label2, _ex_k2, _ex_n2, _ex_m2, _ex_su2, _ex_freq2 = _stem_example
        _ex_this_nt2 = _ex_m2 * _ex_su2
        _worked_example_stem = (
            f"e.g. this report's own rank {html.escape(str(_ex_rr2['rank']))} candidate, "
            f"{html.escape(_ex_label2)} state (stem_up = {_ex_su2} nt): {_kofn(_ex_k2, _ex_n2)} "
            f"samples individually reached the {OPEN_THRESHOLD:.0%} threshold "
            f"({_ex_freq2:.1%} of samples), while the MEAN across all {_ex_n2} samples is "
            f"{_pct(_ex_m2)} -- on average {_ex_this_nt2:.1f}/{_ex_su2} nt of stem_up's own nt "
            f"were paired to stem_down, averaged ACROSS ALL {_ex_n2} samples. That {_pct(_ex_m2)} "
            f"mean and that {_ex_freq2:.1%} k/N frequency are two different statistics about the "
            f"same {_ex_n2} samples, and they do not agree here by design."
        )
    else:
        _worked_example_stem = ""

    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>Single-switch ON/OFF report</title><style>{_REPORT_CSS}</style></head>
    <body class="scr"><div class="scr-wrap">
      <p class="scr-eyebrow">CERNAL · eukaryotic toehold switch</p>
      <h1>Single-switch ON/OFF report</h1>
      <p class="scr-dek">Source: <span class="scr-mono">{html.escape(CSV_PATH)}</span>.
        {SINGLE_POOL_N} candidates screened, top {len(rows)} shown, ranked by ON/OFF
        ratio. Sampling: {N_SAMPLES} Boltzmann draws per state at {TEMPERATURE_C:.1f}
        &deg;C, open threshold {OPEN_THRESHOLD:.0%} of the footprint (toehold+stem_up)
        paired to an external strand. OFF is the worst badness across {{alone, mean of
        up to {SINGLE_N_FAKES} off-target fakes}} -- see the Glossary below for exactly
        what "badness" means and how it is computed. ON/OFF percentages are shown with
        the underlying nt-count and sample count behind them (e.g. "100.0% (27.0/27 nt,
        n={N_SAMPLES})"). Ratio = ON% / (1 + OFF%), in percentage-point units.</p>
      <section>
        <h2>Glossary</h2>
        <div class="scr-table-wrap"><table>
          <tbody>
            <tr><td style="width:130px"><b>ON</b></td><td>fraction of Boltzmann-sampled
              structures (of N_SAMPLES draws) where the REAL trigger is actually bound to
              the footprint (paired to an external strand, &ge;OPEN_THRESHOLD of the
              footprint) -- the frac_open frequency of the +real-trigger state.</td></tr>
            <tr><td><b>OFF</b></td><td>the WORST <i>badness</i> across every illegitimate
              state checked (alone, mean over fake triggers) -- never "1 - ON". Badness =
              the worse of (a) trigger-binding leak (frac_open) and (b) stem-loosening leak
              (the FREQUENCY of samples where stem_up failed to clear OPEN_THRESHOLD as
              closed, <code>1 - n_stem_closed/n_samples</code>) -- a state can leak either
              way, and OFF must catch both, not just whichever is easiest to measure
              (2026-10-05 fix; parallels this spec's own Trap 5). Every candidate's own card
              states which of the two actually set its OFF.</td></tr>
            <tr><td><b>ratio</b></td><td>ON% / (1 + OFF%), in percentage-point units --
              higher is better; OFF=0 still gives a clean, finite value.</td></tr>
            <tr><td><b>dG</b></td><td>ensemble free-energy difference (switch+trigger minus
              switch alone), kcal/mol -- more negative means trigger binding stabilises the
              complex more.</td></tr>
            <tr><td><b>FAKES</b></td><td>an off-target check: a trigger NOT meant for this
              switch (screened by sequence complementarity, not transcript position),
              checked for whether it accidentally opens the switch anyway. Shown as the mean
              nt of stem_up still bound to stem_down across up to {SINGLE_N_FAKES} fake-trigger
              draws, WITH that mean expressed as a % of stem_up right next to it (e.g. "6.8/18
              nt (37.8%)") -- the ideal is the FULL stem_up length (100%), so a reader can
              directly compare the shown % against OPEN_THRESHOLD without doing the division
              themselves. "--" means no qualifying fake could be found for this candidate
              under the current overlap/wobble screen (see the note in that candidate's own
              card), not a failed measurement.</td></tr>
            <tr><td><b>trigger_bound</b></td><td>fraction of the footprint (toehold+stem_up)
              paired to an EXTERNAL strand in a given sampled structure -- the actual ON
              signal. A fraction of the FOOTPRINT length.</td></tr>
            <tr><td><b>stem_closed</b></td><td>fraction of stem_up paired specifically to
              stem_down -- the intended OFF hairpin actually shut. A fraction of the STEM_UP
              length alone (not the whole footprint). Its "k/N (X% of samples &middot; mean Y%)"
              display means specifically: <b>mean</b> is the average, ACROSS ALL N
              Boltzmann-sampled structures, of the PER-SAMPLE fraction of stem_up's own nt that
              are paired to stem_down IN THAT ONE sampled structure -- e.g. if stem_up is 18 nt
              and one given sample has 15 of those 18 nt paired to stem_down, that sample's own
              fraction is 15/18 = 83%; "mean" is the average of that per-sample percentage over
              all N samples. This is a DIFFERENT number from "how many samples individually
              reached the {OPEN_THRESHOLD:.0%} threshold" (the k/N part, literally k&divide;N) --
              the two need not agree, and in general will not. {_worked_example_stem}</td></tr>
            <tr><td><b>misfolded</b></td><td>fraction of the footprint paired to something
              that is neither an external strand nor its own stem_down -- a structural risk
              signal, not a measured expression leak. A fraction of the FOOTPRINT length,
              same base as trigger_bound.</td></tr>
            <tr><td><b>open</b></td><td>a sampled structure where trigger_bound &ge;
              OPEN_THRESHOLD.</td></tr>
            <tr><td><b>k/N</b></td><td>k of the N Boltzmann samples drawn crossed a given
              threshold -- the percentage shown next to it is always literally k&divide;N.
              The continuous MEAN shown alongside is a different, complementary number: the
              average amount of hybridisation across all N samples, not how often it crosses
              the threshold -- the two will not generally agree, by design. {_worked_example}</td></tr>
            <tr><td><b>unique structures</b></td><td>how many of the N Boltzmann draws for a
              state were distinct dot-bracket structures, not repeats -- a low unique count
              relative to N means the sampled ensemble is concentrated on a few
              conformations, so more draws would likely not find much new.</td></tr>
          </tbody>
        </table></div>
      </section>
      <section>
        <h2>Summary</h2>
        <div class="scr-table-wrap scr-table-summary"><table>
          <thead><tr><th>#</th><th>rank</th><th class="num">ON</th><th class="num">OFF</th><th class="num">ratio</th>
            <th class="num">ON&minus;OFF</th><th class="num">dG</th><th class="num">FAKES</th><th></th></tr></thead>
          <tbody>{"".join(summary_trs)}</tbody>
        </table></div>
        <p class="scr-p">See the Glossary above for what every column means -- FAKES in
          particular is explained there (column header shortened; full definition moved
          up rather than duplicated here). A "--" FAKES cell means no qualifying fake was
          found for that candidate (see its own card), not a zero measurement.</p>
      </section>
      <section><h2>Candidates</h2>{"".join(cards)}</section>
      <footer class="scr-footer">Generated {datetime.datetime.now().isoformat(timespec="seconds")}
        · OPEN_THRESHOLD={OPEN_THRESHOLD} (not comparable to the predecessor notebook\'s 0.80)
        · FoldEngine {folder.versions()}</footer>
    </div></body></html>"""


_single_html = render_single_report_html(single_report_rows)
_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
single_report_html_path = _report_root / f"single_switch_report_{_ts}.html"
single_report_html_path.write_text(_single_html)
print(f"wrote {single_report_html_path.stat().st_size:,} bytes -> {single_report_html_path}")

import subprocess, sys as _sys
_render_pdf_script = None
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidate = _base / "src/engine/gates/notebooks/toehold/render_pdf.py"
    if _candidate.exists():
        _render_pdf_script = _candidate
        break
if _render_pdf_script is None:
    print("render_pdf.py not found -- HTML written, PDF skipped")
    single_report_pdf_path = None
else:
    single_report_pdf_path = single_report_html_path.with_suffix(".pdf")
    subprocess.run([_sys.executable, str(_render_pdf_script),
                     str(single_report_html_path), str(single_report_pdf_path)], check=True)
    print(f"wrote {single_report_pdf_path}")


# Part 2 — AND (two inputs)

`AND_GRID_N` top singles (by single-switch ON/OFF ratio) fused into every ordered pair
(self-pairs excluded), both spacer variants. Each pair's 8 states (alone, +A, +B, +A+B,
+fakeA mean of up to `AND_N_FAKES`, +fakeB mean, +A+fakeA mean, +B+fakeB mean) are
decomposed the same three-way way as Part 1, from one shared set of sampled structures
per state, with "both open" read PER STRUCTURE (never from two separately-counted
marginals -- the predecessor notebook's own `JointStateJob` exists for exactly this
reason). Output: `var/reports/and_report_<timestamp>.html/.pdf`.


In [12]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # BENCHMARK FIRST (and_eu_report_spec.md Step 3.5): one real pair's actual job
    # count and wall time, BEFORE committing to the full AND_GRID_N x AND_GRID_N grid
    # -- the spec's own cost section warns AND_N_FAKES=10/side can mean up to 44
    # jobs/pair (4 base + 4 fake-involving states x 10 draws). In practice the real
    # fake-availability (Trap 2/3 screening is strict) means many pairs find FEWER
    # than 10 qualifying fakes per side, so the real job count is usually well under
    # the worst case -- this cell measures the real number, not the worst case, on
    # the actual top-AND_GRID_N pool.
    # ==============================================================================
    and_pool_designs = [r["design"] for r in single_scored[:AND_GRID_N]]
    print(f"AND pool: top {len(and_pool_designs)} singles by single-switch ON/OFF ratio "
          f"(ranks {[d['rank'] for d in and_pool_designs]})")

    _bd5, _bd3 = and_pool_designs[0], and_pool_designs[1 % len(and_pool_designs)]
    _bbuilt, _bproblem = fuse(_bd5, _bd3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER, spacer="")
    assert _bproblem is None, _bproblem

    _bfp5, _bsu5, _bsd5 = _bbuilt["footprint_5p"], _bbuilt["domains_5p"]["stem_up"], _bbuilt["domains_5p"]["stem_down"]
    _bfp3, _bsu3, _bsd3 = _bbuilt["footprint_3p"], _bbuilt["domains_3p"]["stem_up"], _bbuilt["domains_3p"]["stem_down"]

    def _mk_and_job(key, extra, built=_bbuilt):
        return AndDecompJob(
            key=key, fused=built["fused"], extra=extra,
            footprint_a=_bfp5, stem_up_a=_bsu5, stem_down_a=_bsd5,
            footprint_b=_bfp3, stem_up_b=_bsu3, stem_down_b=_bsd3,
            temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
        )

    _b_fakes_a, _picked_a = [], {_bd5["rank"], _bd3["rank"]}
    for _k in range(AND_N_FAKES):
        _wd = pick_fake_design(_bd5, _bd3, _picked_a, and_pool_designs, start_offset=_k,
                               max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT, wobble_max_fraction=WOBBLE_MAX_FRACTION)
        if _wd is None:
            break
        _picked_a.add(_wd["rank"])
        _b_fakes_a.append(_wd)
    _b_fakes_b, _picked_b = [], {_bd5["rank"], _bd3["rank"]}
    for _k in range(AND_N_FAKES):
        _wd = pick_fake_design(_bd5, _bd3, _picked_b, and_pool_designs, start_offset=_k + 1000,
                               max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT, wobble_max_fraction=WOBBLE_MAX_FRACTION)
        if _wd is None:
            break
        _picked_b.add(_wd["rank"])
        _b_fakes_b.append(_wd)

    _bjobs = [_mk_and_job("alone", ()), _mk_and_job("+A", (_bd5["trigger"],)),
              _mk_and_job("+B", (_bd3["trigger"],)), _mk_and_job("+A+B", (_bd5["trigger"], _bd3["trigger"]))]
    for _i, _wd in enumerate(_b_fakes_a):
        _bjobs.append(_mk_and_job(f"+fakeA{_i}", (_wd["trigger"],)))
        _bjobs.append(_mk_and_job(f"+A+fakeA{_i}", (_bd5["trigger"], _wd["trigger"])))
    for _i, _wd in enumerate(_b_fakes_b):
        _bjobs.append(_mk_and_job(f"+fakeB{_i}", (_wd["trigger"],)))
        _bjobs.append(_mk_and_job(f"+B+fakeB{_i}", (_bd3["trigger"], _wd["trigger"])))

    print(f"ONE PAIR ({_bd5['rank']}+{_bd3['rank']}): fakeA pool {len(_b_fakes_a)}/{AND_N_FAKES}, "
          f"fakeB pool {len(_b_fakes_b)}/{AND_N_FAKES} -> {len(_bjobs)} jobs "
          f"(worst case would be {4 + 4 * AND_N_FAKES})")
    _bt0 = time.time()
    run_parallel_and_decomp(_bjobs, max_workers=MAX_WORKERS)
    _b_elapsed = time.time() - _bt0
    _per_job_s = _b_elapsed / len(_bjobs)
    print(f"ONE PAIR: {_b_elapsed:.2f}s for {len(_bjobs)} jobs ({_per_job_s * 1000:.1f} ms/job wall, "
          f"N_SAMPLES={N_SAMPLES}, MAX_WORKERS={MAX_WORKERS})")

    _n_grid_pairs = AND_GRID_N * (AND_GRID_N - 1)  # ordered, self-pairs excluded
    _n_spacer = len(SPACER_VARIANTS)
    _worst_jobs_per_pair = 4 + 4 * AND_N_FAKES
    _projected_worst_s = _n_grid_pairs * _n_spacer * _worst_jobs_per_pair * _per_job_s
    _projected_measured_s = _n_grid_pairs * _n_spacer * len(_bjobs) * _per_job_s
    print(f"\nPROJECTION for AND_GRID_N={AND_GRID_N}, {_n_spacer} spacer variants: "
          f"{_n_grid_pairs} pairs x {_n_spacer} = {_n_grid_pairs * _n_spacer} (pair, spacer) combos")
    print(f"  worst case ({_worst_jobs_per_pair} jobs/combo, full {AND_N_FAKES}/{AND_N_FAKES} fakes "
          f"every time): ~{_projected_worst_s:.0f}s (~{_projected_worst_s / 60:.1f} min) of real fold time")
    print(f"  measured-this-pair case ({len(_bjobs)} jobs/combo): "
          f"~{_projected_measured_s:.0f}s (~{_projected_measured_s / 60:.1f} min)")
    print("  (actual grid cost will differ per-pair since fake availability varies --")
    print("   the full grid cell below prints the REAL total once it finishes.)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the AND benchmark.")


SNAPSHOT_TO_LOAD is set -- skipping the AND benchmark.


In [13]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # AND GRID -- AND_GRID_N x itself, self-pairs excluded, both spacer variants.
    # Every job of every pair of every spacer variant is built first, then submitted
    # as ONE list to ONE run_parallel_and_decomp call (CLAUDE.md sec 5 / Trap 6).
    # ==============================================================================
    SPACER_CANDIDATES = generate_spacer_candidates(
        SPACER_CANDIDATE_POOL_SIZE, SPACER_MIN_LENGTH, SPACER_MAX_LENGTH, seed=SPACER_SEED,
    )

    and_jobs = []
    and_built_by_key = {}  # key -> (d5, d3, built, spacer_label, spacer_seq, fakes_a, fakes_b)
    _and_t0 = time.time()
    _n_excluded_self = 0

    for _spacer_label in SPACER_VARIANTS:
        for _d5 in and_pool_designs:
            for _d3 in and_pool_designs:
                if _d5["rank"] == _d3["rank"]:
                    _n_excluded_self += 1
                    continue
                if _spacer_label == "spacer":
                    _spacer_seq, _spacer_worst = pick_optimized_spacer(_d5, _d3, SPACER_CANDIDATES)
                else:
                    _spacer_seq, _spacer_worst = "", None
                _built, _problem = fuse(_d5, _d3, cut_before=CUT_5P_BEFORE, keep_after=KEEP_3P_AFTER,
                                        spacer=_spacer_seq)
                if _problem is not None:
                    continue

                _fp5, _su5, _sd5 = _built["footprint_5p"], _built["domains_5p"]["stem_up"], _built["domains_5p"]["stem_down"]
                _fp3, _su3, _sd3 = _built["footprint_3p"], _built["domains_3p"]["stem_up"], _built["domains_3p"]["stem_down"]

                def _mkjob(key, extra, built=_built, fp5=_fp5, su5=_su5, sd5=_sd5, fp3=_fp3, su3=_su3, sd3=_sd3):
                    return AndDecompJob(
                        key=key, fused=built["fused"], extra=extra,
                        footprint_a=fp5, stem_up_a=su5, stem_down_a=sd5,
                        footprint_b=fp3, stem_up_b=su3, stem_down_b=sd3,
                        temperature=TEMPERATURE_C, n_samples=N_SAMPLES, threshold=OPEN_THRESHOLD,
                    )

                _pair_key = f"{_d5['rank']}+{_d3['rank']}|{_spacer_label}"
                _picked_a = {_d5["rank"], _d3["rank"]}
                _fakes_a = []
                for _k in range(AND_N_FAKES):
                    _wd = pick_fake_design(_d5, _d3, _picked_a, and_pool_designs, start_offset=_k,
                                           max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                           wobble_max_fraction=WOBBLE_MAX_FRACTION)
                    if _wd is None:
                        break
                    _picked_a.add(_wd["rank"])
                    _fakes_a.append(_wd)
                _picked_b = {_d5["rank"], _d3["rank"]}
                _fakes_b = []
                for _k in range(AND_N_FAKES):
                    _wd = pick_fake_design(_d5, _d3, _picked_b, and_pool_designs, start_offset=_k + 1000,
                                           max_overlap_nt=FAKE_MAX_SEQ_OVERLAP_NT,
                                           wobble_max_fraction=WOBBLE_MAX_FRACTION)
                    if _wd is None:
                        break
                    _picked_b.add(_wd["rank"])
                    _fakes_b.append(_wd)

                and_built_by_key[_pair_key] = (_d5, _d3, _built, _spacer_label, _spacer_seq,
                                                _spacer_worst, _fakes_a, _fakes_b)
                and_jobs.append(_mkjob(f"{_pair_key}|alone", ()))
                and_jobs.append(_mkjob(f"{_pair_key}|+A", (_d5["trigger"],)))
                and_jobs.append(_mkjob(f"{_pair_key}|+B", (_d3["trigger"],)))
                and_jobs.append(_mkjob(f"{_pair_key}|+A+B", (_d5["trigger"], _d3["trigger"])))
                for _i, _wd in enumerate(_fakes_a):
                    and_jobs.append(_mkjob(f"{_pair_key}|+fakeA{_i}", (_wd["trigger"],)))
                    and_jobs.append(_mkjob(f"{_pair_key}|+A+fakeA{_i}", (_d5["trigger"], _wd["trigger"])))
                for _i, _wd in enumerate(_fakes_b):
                    and_jobs.append(_mkjob(f"{_pair_key}|+fakeB{_i}", (_wd["trigger"],)))
                    and_jobs.append(_mkjob(f"{_pair_key}|+B+fakeB{_i}", (_d3["trigger"], _wd["trigger"])))

    print(f"{len(and_built_by_key)} (pair, spacer) combos, {len(and_jobs)} jobs total "
          f"({_n_excluded_self} self-pairs excluded) -- ONE pool.map call")
    and_decomp_by_key = run_parallel_and_decomp(and_jobs, max_workers=MAX_WORKERS)
    _and_elapsed = time.time() - _and_t0
    print(f"AND grid done in {_and_elapsed:.0f}s ({_and_elapsed / 60:.1f} min), "
          f"{_and_elapsed / len(and_jobs) * 1000:.1f} ms/job")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping the AND grid.")


SNAPSHOT_TO_LOAD is set -- skipping the AND grid.


In [ ]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ON/OFF RATIO for AND (and_eu_report_spec.md Step 2 / Trap 5):
    #   ON  = P(both open | +A+B) -- frac_both_open of the "+A+B" state.
    #   OFF = max over badness({alone, +A, +B, mean(+fakeA), mean(+fakeB),
    #         mean(+A+fakeA), mean(+B+fakeB)}) -- never "1 - ON" (Trap 5). badness
    #         (2026-10-05, OFF-broadening) is max(trigger-binding leak, either
    #         side's stem failing to stay closed, and -- for the states where it
    #         applies -- the OTHER hairpin opening without its own trigger present,
    #         i.e. cross-talk). `_and_badness` in cell 7, shared with the
    #         snapshot-restore path (cell 9).
    #
    # ratio = on_pct / (1 + off_pct), percentage-POINT units -- same formula, same
    # reasoning, as the single-switch cell above (fix #3): OFF=0 gives a clean,
    # finite ratio with no floor/bound notation needed.
    # ==============================================================================
    and_scored = []
    for _key, (_d5, _d3, _built, _spacer_label, _spacer_seq, _spacer_worst, _fakes_a, _fakes_b) in and_built_by_key.items():
        _alone = and_decomp_by_key[f"{_key}|alone"]
        _a = and_decomp_by_key[f"{_key}|+A"]
        _b = and_decomp_by_key[f"{_key}|+B"]
        _both = and_decomp_by_key[f"{_key}|+A+B"]
        _fakeA_results = [and_decomp_by_key[f"{_key}|+fakeA{_i}"] for _i in range(len(_fakes_a))]
        _fakeB_results = [and_decomp_by_key[f"{_key}|+fakeB{_i}"] for _i in range(len(_fakes_b))]
        _a_fakeA_results = [and_decomp_by_key[f"{_key}|+A+fakeA{_i}"] for _i in range(len(_fakes_a))]
        _b_fakeB_results = [and_decomp_by_key[f"{_key}|+B+fakeB{_i}"] for _i in range(len(_fakes_b))]

        def _mean_field(results, field):
            return sum(r[field] for r in results) / len(results) if results else None

        def _sum_field(results, field):
            return sum(r[field] for r in results) if results else None

        def _pooled_state(results):
            return {
                "frac_both_open": _mean_field(results, "frac_both_open"),
                "frac_a_open": _mean_field(results, "frac_a_open"),
                "frac_b_open": _mean_field(results, "frac_b_open"),
                "n_stem_closed_a": _sum_field(results, "n_stem_closed_a"),
                "n_stem_closed_b": _sum_field(results, "n_stem_closed_b"),
                "n_samples": _sum_field(results, "n_samples"),
                "mean_stem_closed_a": _mean_field(results, "mean_stem_closed_a"),
                "mean_stem_closed_b": _mean_field(results, "mean_stem_closed_b"),
            }

        # (label, badness, mean_stem_closed_a, mean_stem_closed_b, criterion) for
        # every illegitimate state, built by `_and_off_term` (cell 7) -- mask_side
        # suppresses a side's own stem-loosening term exactly where that side's own
        # REAL trigger is legitimately present in the state (+A/+A+fakeA mask "a";
        # +B/+B+fakeB mask "b"; alone/+fakeA/+fakeB mask neither, since no real
        # trigger is present on either side there at all). Without this mask, a
        # correctly-working design's own intended trigger binding -- which
        # necessarily displaces that side's stem_up from stem_down -- would score
        # as "stem-loosening leak" on every state with any real trigger bound,
        # collapsing OFF to ~100% uniformly (verified directly while building this).
        _off_candidates = [
            _and_off_term("alone", _alone, mask_side=None),
            _and_off_term("+A", _a, mask_side="a"),
            _and_off_term("+B", _b, mask_side="b"),
        ]
        for _label, _results, _mask in (
            ("+fakeA", _fakeA_results, None), ("+fakeB", _fakeB_results, None),
            ("+A+fakeA", _a_fakeA_results, "a"), ("+B+fakeB", _b_fakeB_results, "b"),
        ):
            if not _results:
                continue
            _off_candidates.append(_and_off_term(_label, _pooled_state(_results), mask_side=_mask))
        _off_source, _off, _off_mean_sc_a, _off_mean_sc_b, _off_criterion, _off_terms = max(
            _off_candidates, key=lambda t: t[1])

        # Per-side fake-check OFF nt-count (fix #4, per side -- fakeA checked against
        # hairpin A's own stem, fakeB against hairpin B's): mean of mean_stem_closed_*
        # over that side's fake draws, same averaging shape as the ratio's own fake
        # terms, pure display arithmetic on numbers already computed.
        _fakeA_mean_stem_closed = _mean_field(_fakeA_results, "mean_stem_closed_a")
        _fakeB_mean_stem_closed = _mean_field(_fakeB_results, "mean_stem_closed_b")

        _on = _both["frac_both_open"]
        _on_pct, _off_pct = _on * 100.0, _off * 100.0
        _ratio = _on_pct / (1.0 + _off_pct)

        and_scored.append({
            "key": _key, "d5": _d5, "d3": _d3, "built": _built,
            "spacer_label": _spacer_label, "spacer_seq": _spacer_seq, "spacer_worst": _spacer_worst,
            "fakes_a": _fakes_a, "fakes_b": _fakes_b,
            "alone": _alone, "state_a": _a, "state_b": _b, "both": _both,
            "fakeA_results": _fakeA_results, "fakeB_results": _fakeB_results,
            "a_fakeA_results": _a_fakeA_results, "b_fakeB_results": _b_fakeB_results,
            "fakeA_mean_stem_closed": _fakeA_mean_stem_closed,
            "fakeB_mean_stem_closed": _fakeB_mean_stem_closed,
            "off_source": _off_source, "off_criterion": _off_criterion, "off_terms": _off_terms,
            "off_mean_stem_closed_a": _off_mean_sc_a,
            "off_mean_stem_closed_b": _off_mean_sc_b,
            "on": _on, "off": _off, "ratio": _ratio,
            "on_minus_off": _on - _off,
        })

    and_scored.sort(key=lambda r: -r["ratio"])
    and_report_rows = and_scored[:AND_REPORT_N]
    print(f"{len(and_scored)} (pair, spacer) combos scored; top {len(and_report_rows)} go to the report")
    if and_scored:
        _top = and_scored[0]
        print(f"  best: {_top['d5']['rank']}+{_top['d3']['rank']} [{_top['spacer_label']}] "
              f"ratio={_top['ratio']:.1f}, on={_top['on']:.1%}, off={_top['off']:.1%} "
              f"(off_source={_top['off_source']}, off_criterion={_top['off_criterion']})")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping AND scoring.")


In [15]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # ENERGIES for AND, top AND_REPORT_N only: alone, trigger A alone, trigger B
    # alone, +A, +B, +A+B, and Delta(ON) = ee_both - min(ee_off, ee_tA, ee_tB) (more
    # negative is better) -- measure()'s own d_ee_worst_alt IS this quantity, reused
    # as-is rather than recomputed (CLAUDE.md sec 1).
    # ==============================================================================
    for _r in and_report_rows:
        _d5, _d3, _built = _r["d5"], _r["d3"], _r["built"]
        _dg = measure(_built, _d5["trigger"], _d3["trigger"], folder,
                      _d3["metrics"]["own_kozak_rc_score"], _d5["metrics"]["own_prefix_rc_score"])
        _r["dg"] = _dg
        _r["ee_trigger_a"] = folder.partition(_d5["trigger"])
        _r["ee_trigger_b"] = folder.partition(_d3["trigger"])
    print(f"energies computed for {len(and_report_rows)} AND report candidates")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping AND energies.")


SNAPSHOT_TO_LOAD is set -- skipping AND energies.


In [ ]:
# ==================================================================================
# CROSS-DOMAIN BINDING (2026-10-05, AND item 6; extended AND item 3 same round) --
# NEW FIELD, display-only diagnostic, no ranking input. The snapshot never stored raw
# per-sample structures (see _joint_state_parallel.py's own docstring on why sampling
# isn't cached to disk), so this needs a real, fresh re-sample -- narrowly backfilled
# for only `and_report_rows` (~20 candidates), same pattern as the n_stem_closed/
# n_misfolded backfill above, never the full grid. Computes, for a given state: mean
# fraction of spacer nt paired to EITHER hairpin's own toehold, and mean fraction of
# each hairpin's toehold paired to the OTHER hairpin's prefix -- unwanted cross-domain
# competition, not the intended trigger binding (already reported elsewhere as
# trigger_bound_a/b). `_cross_domain_worker` reuses `_partner_table`, the one
# stack-walk every other worker in this module already shares (CLAUDE.md sec 1) -- no
# second pairing walker written here.
#
# AND item 3 fix (2026-10-05): this metric used to cover ONLY the "+A+B" (ON) state,
# which reads near-zero for every candidate essentially by construction -- the real
# trigger binding in ON displaces exactly the cross-domain pairing this metric is
# trying to catch. A real, substantial cross-domain interaction (confirmed directly,
# e.g. this report's own #1 candidate: hairpin A's toehold paired to the spacer, and
# hairpin B's toehold paired to hairpin A's prefix, in the real MFE "alone" structure)
# was therefore always invisible in this metric even though it is plainly visible in
# the "alone" structure plot right next to it. Both states are now sampled through the
# SAME CrossDomainJob/_cross_domain_worker (`include_triggers=False` for "alone"), not
# a second implementation -- and both are shown, clearly labelled by state, so a reader
# comparing the metric against the alone-state plot is comparing like with like.
# ==================================================================================
_cross_jobs = []
for _r in and_report_rows:
    if "mean_toeholdA_to_prefixB" in _r:
        continue
    _built = _r["built"]
    _spacer_span = _built.get("spacer_span")
    _spacer_span = tuple(_spacer_span) if _spacer_span and _spacer_span[1] > _spacer_span[0] else None
    # prefix_b is routinely absent for this topology: KEEP_3P_AFTER="prefix" means
    # fuse() cuts the 3' half's own prefix domain away entirely (it never survives
    # into `built["domains_3p"]`) -- `.get()` + None, not a KeyError or an assumed
    # span, matches CrossDomainJob's own None convention (CLAUDE.md sec 3).
    _prefix_a = _built["domains_5p"].get("prefix")
    _prefix_b = _built["domains_3p"].get("prefix")

    def _mk_cross_job(state_suffix, include_triggers, _r=_r, _built=_built, _spacer_span=_spacer_span,
                       _prefix_a=_prefix_a, _prefix_b=_prefix_b):
        return CrossDomainJob(
            key=f"{_r['key']}|{state_suffix}", fused=_built["fused"],
            trigger_a=_r["d5"]["trigger"], trigger_b=_r["d3"]["trigger"],
            toehold_a=tuple(_built["domains_5p"]["toehold"]), toehold_b=tuple(_built["domains_3p"]["toehold"]),
            prefix_a=tuple(_prefix_a) if _prefix_a else None,
            prefix_b=tuple(_prefix_b) if _prefix_b else None,
            spacer=_spacer_span, temperature=TEMPERATURE_C, n_samples=N_SAMPLES,
            include_triggers=include_triggers,
        )

    _cross_jobs.append(_mk_cross_job("+A+B", True))
    _cross_jobs.append(_mk_cross_job("alone", False))

if _cross_jobs:
    _cdt0 = time.time()
    _cross_results = run_parallel_cross_domain(_cross_jobs, max_workers=MAX_WORKERS)
    for _r in and_report_rows:
        _on_key, _alone_key = f"{_r['key']}|+A+B", f"{_r['key']}|alone"
        if _on_key in _cross_results:
            _fresh = _cross_results[_on_key]
            _r["mean_spacer_cross_toehold"] = _fresh["mean_spacer_cross_toehold"]
            _r["mean_toeholdA_to_prefixB"] = _fresh["mean_toeholdA_to_prefixB"]
            _r["mean_toeholdB_to_prefixA"] = _fresh["mean_toeholdB_to_prefixA"]
        if _alone_key in _cross_results:
            _fresh_alone = _cross_results[_alone_key]
            _r["alone_mean_spacer_cross_toehold"] = _fresh_alone["mean_spacer_cross_toehold"]
            _r["alone_mean_toeholdA_to_prefixB"] = _fresh_alone["mean_toeholdA_to_prefixB"]
            _r["alone_mean_toeholdB_to_prefixA"] = _fresh_alone["mean_toeholdB_to_prefixA"]
    print(f"cross-domain binding backfilled for {len(and_report_rows)} AND report rows "
          f"in {time.time() - _cdt0:.1f}s (+A+B AND alone states, display-only, no ranking effect)")
else:
    print("and_report_rows already carry the cross-domain-binding fields -- no backfill needed")


In [ ]:
# Images -- UNGATED (always runs, same rationale as the single-switch images cell).
#
# 2026-10-05 fix (AND item 4): every structure plot below used to draw the FULL
# `_fused` (and `_fused` + triggers) sequence, including whatever payload/trailing
# domain survives past the last hairpin's own kozak domain (e.g. hairpin B's
# exp_gene, which `KEEP_3P_AFTER="prefix"` keeps attached) -- the previous round's
# "drop exp_gene from the plot" fix only removed exp_gene's domain_fill COLOR entry
# (cell 7's `_PAYLOAD_DOMAINS`), it never stopped those bases from being PLOTTED, so
# they still rendered, just in the flat default-gray fill -- exactly the "unexplained
# gray circles after the kozak domain" the user saw, worst on the "alone"/OFF plot
# since that one has no trigger-hiding logic to coincidentally also crop them.
# Fixed by reusing the SAME `visible_len` clipping mechanism `structure_png` already
# has for hiding a hidden trigger strand (cell 7) -- applied here to clip every AND
# structure plot (alone/+A/+B/+A+B alike, not just +A+B) to the end of the LAST real
# kozak domain actually present in the fused frame (whichever hairpin keeps one after
# fuse() -- for this topology that is hairpin B's). Any real base pair whose partner
# falls outside that range still gets the existing dangling-ring treatment, so a real
# bond into the hidden tail or into a trigger strand is never silently dropped.
for _r in and_report_rows:
    _built = _r["built"]
    _fused = _built["fused"]
    _fp5, _fp3 = _built["footprint_5p"], _built["footprint_3p"]
    _spans2 = [(_fp5[0], _fp5[1], _COLOR_A), (_fp3[0], _fp3[1], _COLOR_B)]

    # Real per-domain fill color for BOTH hairpins (fix #6) -- a doubled palette with
    # a " (A)"/" (B)" label suffix so the legend tells hairpin 1's "toehold" apart
    # from hairpin 2's, instead of one color silently winning a name collision.
    # 2026-10-05 fix (item 12): exp_gene/opt_exp_gene excluded via the shared
    # _PAYLOAD_DOMAINS (cell 7) -- same rationale as the single-switch images cell;
    # the payload domain itself is untouched in `_built`, only what gets drawn here.
    _kept5p = {n: s for n, s in _built["domains_5p"].items() if n not in _PAYLOAD_DOMAINS}
    _kept3p = {n: s for n, s in _built["domains_3p"].items() if n not in _PAYLOAD_DOMAINS}
    _domain_fill_and = (
        _domain_fill_from_domains(_kept5p, _DOMAIN_COLOR_A, " (A)")
        + _domain_fill_from_domains(_kept3p, _DOMAIN_COLOR_B, " (B)")
    )
    if _built.get("spacer_span"):
        _sp0, _sp1 = _built["spacer_span"]
        if _sp1 > _sp0:
            # 2026-10-05 fix (item 14): "spacer" is now a real key in both domain
            # palettes (_SPACER_COLOR) rather than a hardcoded, nearly-invisible ad
            # hoc color here.
            _domain_fill_and.append((_sp0, _sp1, _SPACER_COLOR, "spacer"))

    # End of the last real (non-zero-length) "kozak" domain surviving in the fused
    # frame, over EITHER hairpin -- whichever half actually keeps one after fuse().
    # Falls back to len(_fused) (no clipping at all) if, for some topology, neither
    # half has one, rather than guessing a cutoff.
    _kozak_ends_and = [
        _e for _kd in (_kept5p, _kept3p)
        for _n, (_s, _e) in _kd.items() if _n == "kozak" and _e > _s
    ]
    _visible_len_and = max(_kozak_ends_and) if _kozak_ends_and else len(_fused)

    _mfe_alone = folder.mfe(_fused)
    _matrix_alone = folder.base_pair_probabilities(_fused)
    _r["structure_alone_png"] = structure_png(
        _fused, _mfe_alone.structure,
        f"{_r['d5']['rank']}+{_r['d3']['rank']} [{_r['spacer_label']}] — alone (OFF)",
        spans=_spans2, domain_fill=_domain_fill_and, visible_len=_visible_len_and,
    )
    # No strand present besides the fused construct itself -- no junction to mark
    # (fix #7).
    _r["matrix_alone_png"] = bpp_heatmap_png(_fused, _matrix_alone, "alone: base-pair probabilities")

    # One crosshair per real strand junction actually present in THIS state's matrix
    # (fix #7, AND version): +A/+B have exactly one (switch<->that trigger, in its own
    # color); +A+B has two (switch<->triggerA, then triggerA<->triggerB), each in its
    # own strand's color so two boundaries in one matrix stay distinguishable.
    _STATE_BOUNDARY_COLORS = {
        "+A": (_COLOR_A,),
        "+B": (_COLOR_B,),
        "+A+B": (_COLOR_A, _COLOR_B),
    }
    for _state_name, _extra in (("+A", (_r["d5"]["trigger"],)), ("+B", (_r["d3"]["trigger"],)),
                                 ("+A+B", (_r["d5"]["trigger"], _r["d3"]["trigger"]))):
        _strands = "&".join([_fused, *_extra])
        _concat = _fused + "".join(_extra)
        _mfe = folder.mfe(_strands)
        _matrix = folder.base_pair_probabilities(_strands)
        # 2026-10-05 fix (AND item 7, re-fixed AND item 4 same round): "+A+B" (ON)
        # already hid both trigger strands via visible_len=len(_fused) -- that bound
        # also now needs to be the SAME kozak-end clip as "alone" above (not just
        # len(_fused)), or the payload tail past kozak would still render uncolored
        # past where the triggers used to start. "+A"/"+B" previously drew their one
        # trigger with NO clipping at all (visible_len=None) -- now also clipped to
        # the same kozak-end boundary, which hides that trigger too (it sits past the
        # clip point) with the existing dangling-ring marker standing in for it, same
        # as any other real pair into a hidden region.
        _r[f"structure_{_state_name}_png"] = structure_png(
            _concat, _mfe.structure, f"{_r['d5']['rank']}+{_r['d3']['rank']} — {_state_name}",
            spans=_spans2, domain_fill=_domain_fill_and, visible_len=_visible_len_and,
        )
        _boundary_positions = []
        _pos = len(_fused)
        for _strand, _color in zip(_extra, _STATE_BOUNDARY_COLORS[_state_name], strict=True):
            _boundary_positions.append((_pos, _color))
            _pos += len(_strand)
        _r[f"matrix_{_state_name}_png"] = bpp_heatmap_png(
            _concat, _matrix, f"{_state_name}: base-pair probabilities",
            boundaries=_boundary_positions,
        )

    # Domain diagram: both hairpins in one fused-frame bar, doubled palette (A-tinted
    # vs B-tinted) so hairpin 1 and 2 are distinguishable (and_eu_report_spec.md Step 4).
    # 2026-10-05 fix (item 12): built from _kept5p/_kept3p (exp_gene already excluded
    # above), not the raw domains_5p/domains_3p.
    _segments = []
    for _half_domains, _palette in ((_kept5p, _DOMAIN_COLOR_A), (_kept3p, _DOMAIN_COLOR_B)):
        for _name, (_s, _e) in _half_domains.items():
            if _e > _s:
                _segments.append((_s, _name, _e - _s, _palette.get(_name, "#B7C4BC"), _fused[_s:_e]))
    if _built.get("spacer_span"):
        _s0, _s1 = _built["spacer_span"]
        if _s1 > _s0:
            _segments.append((_s0, "spacer", _s1 - _s0, _SPACER_COLOR, _fused[_s0:_s1]))
    # Total fused (two-hairpin) construct length shown on the bar itself (item 13) --
    # excludes both triggers, which are never part of `_fused`.
    _r["layout_png"] = layout_bar_png(
        _segments, len(_fused),
        f"{_r['d5']['rank']}+{_r['d3']['rank']} [{_r['spacer_label']}] — domain layout "
        f"(both hairpins, fused length {len(_fused)} nt)",
    )
    _r["domain_rows"] = [(name, length, seq) for (_s, name, length, _c, seq) in sorted(_segments, key=lambda s: s[0])]

print(f"images built for {len(and_report_rows)} AND candidates")


In [ ]:
def _mean_or_none(results, field):
    vals = [r[field] for r in results]
    return sum(vals) / len(vals) if vals else None


def _sum_or_none(results, field):
    return sum(r[field] for r in results) if results else None


def render_and_report_html(rows):
    summary_trs = []
    for i, r in enumerate(rows, start=1):
        _built = r["built"]
        _su5_len = _built["domains_5p"]["stem_up"][1] - _built["domains_5p"]["stem_up"][0]
        _su3_len = _built["domains_3p"]["stem_up"][1] - _built["domains_3p"]["stem_up"][0]
        _fp5_len = _built["footprint_5p"][1] - _built["footprint_5p"][0]
        _fp3_len = _built["footprint_3p"][1] - _built["footprint_3p"][0]
        # Per-side OFF (fakes) nt-count (fix #4, AND version): fakeA checked against
        # hairpin A's own stem, fakeB against hairpin B's -- two separate numbers, not
        # one combined "fakes found" count. Shown with its own mean % (fix #1,
        # 2026-10-05) via _nt_pct, not a bare nt-count.
        _fakeA_off_nt = _nt_pct(r["fakeA_mean_stem_closed"], _su5_len)
        _fakeB_off_nt = _nt_pct(r["fakeB_mean_stem_closed"], _su3_len)
        # ON/OFF shown PER SIDE (fix #2, AND version: never one combined number for
        # two footprints) -- ON's headline is the joint "both open" frequency (an
        # irreducibly joint event), with each side's own nt-count alongside; OFF's
        # headline is the worst illegitimate state's joint frequency, with that SAME
        # state's own per-side stem-closed nt-count (never a number borrowed from a
        # different state than the one OFF actually came from).
        # Compact (summary row) vs. full (detail-card head) versions of the same
        # per-side nt-count -- the summary table has 9 columns across 20 rows and has
        # to stay on one landscape page, so it drops the "(n=..., state=...)" context
        # the card head keeps (that context is also restated in the card's own decomp
        # table a few lines below it).
        _on_compact = (f'A: {_nt(r["both"]["mean_trigger_bound_a"], _fp5_len)} &middot; '
                       f'B: {_nt(r["both"]["mean_trigger_bound_b"], _fp3_len)}')
        _off_compact = (f'A: {_nt(r["off_mean_stem_closed_a"], _su5_len)} &middot; '
                        f'B: {_nt(r["off_mean_stem_closed_b"], _su3_len)}')
        # 2026-10-05 fix (AND item 1b): the same off_source-with-both-terms
        # breakdown the card head now shows, compacted into a <small> line under the
        # OFF percentage itself -- so the summary table answers "which term, and
        # what was the losing term's own number" without having to open the card.
        _off_breakdown_compact = _off_breakdown_text(r["off_source"], r["off_criterion"], r.get("off_terms"))
        summary_trs.append(
            "<tr>"
            f"<td>{i}</td><td>{html.escape(r['spacer_label'])}</td>"
            f"<td>{html.escape(str(r['d5']['rank']))}+{html.escape(str(r['d3']['rank']))}</td>"
            f'<td class="num">{_pct(r["on"])} ({_on_compact})</td>'
            f'<td class="num">{_pct(r["off"])} ({_off_compact})'
            f'<br><small>{_off_breakdown_compact}</small></td>'
            f'<td class="num">{r["ratio"]:.1f}</td>'
            f'<td class="num">{_pct(r["on_minus_off"])}</td>'
            f'<td class="num">{_num(r["dg"]["d_ee_worst_alt"])}</td>'
            f'<td class="num">A: {_fakeA_off_nt} &middot; B: {_fakeB_off_nt}</td>'
            f'<td><a href="#and-{i}">detail &darr;</a></td>'
            "</tr>"
        )

    cards = []
    for i, r in enumerate(rows, start=1):
        _built = r["built"]
        _fp5_len = _built["footprint_5p"][1] - _built["footprint_5p"][0]
        _fp3_len = _built["footprint_3p"][1] - _built["footprint_3p"][0]
        _su5_len = _built["domains_5p"]["stem_up"][1] - _built["domains_5p"]["stem_up"][0]
        _su3_len = _built["domains_3p"]["stem_up"][1] - _built["domains_3p"]["stem_up"][0]
        # Full per-side ON/OFF detail for the card head (fix #2) -- more room here
        # than the summary row, so this keeps the sample count and the OFF-defining
        # state name the compact summary-row version drops.
        _on_full = (f'A: {_nt(r["both"]["mean_trigger_bound_a"], _fp5_len)} &middot; '
                    f'B: {_nt(r["both"]["mean_trigger_bound_b"], _fp3_len)} (n={r["both"]["n_samples"]})')
        _off_full = (f'A: {_nt(r["off_mean_stem_closed_a"], _su5_len)} &middot; '
                     f'B: {_nt(r["off_mean_stem_closed_b"], _su3_len)} '
                     f'(n={N_SAMPLES}, state={html.escape(r["off_source"])})')
        # 2026-10-05 fix (item 2, AND version, upgraded this round -- AND item 1b):
        # which criterion -- trigger-binding, stem-loosening (A or B), or cross-talk
        # -- actually determined off_source for THIS candidate. Now shows EVERY term
        # considered and its own number (`_off_breakdown_text`, cell 7), not just the
        # winning term's name, per side where applicable.
        _off_note = _off_breakdown_text(r["off_source"], r["off_criterion"], r.get("off_terms"))

        def _state_row(label, state):
            # 2026-10-05 root-bug fix: trigger_bound_a/b now get the SAME k/N
            # treatment as stem_closed/misfolded (previously a bare continuous-mean
            # percentage with no sample count behind it at all, unlike "both open"
            # which at least had a derived count) -- using the exact n_a_open/
            # n_b_open integer counts the worker now returns, never a
            # round(frac * n) reconstruction. "both open" below is fixed the same
            # way: n_both_open is the real per-structure joint count, not a
            # recomputed round().
            return (f"<tr><td>{label}</td>"
                    f'<td class="num">{_kofn_pct(state.get("n_a_open"), state["n_samples"], state["mean_trigger_bound_a"])}</td>'
                    f'<td class="num">{_kofn_pct(state.get("n_stem_closed_a"), state["n_samples"], state["mean_stem_closed_a"])}</td>'
                    f'<td class="num">{_kofn_pct(state.get("n_misfolded_a"), state["n_samples"], state["mean_misfolded_a"])}</td>'
                    f'<td class="num">{_kofn_pct(state.get("n_b_open"), state["n_samples"], state["mean_trigger_bound_b"])}</td>'
                    f'<td class="num">{_kofn_pct(state.get("n_stem_closed_b"), state["n_samples"], state["mean_stem_closed_b"])}</td>'
                    f'<td class="num">{_kofn_pct(state.get("n_misfolded_b"), state["n_samples"], state["mean_misfolded_b"])}</td>'
                    f'<td class="num">{_kofn(state.get("n_both_open"), state["n_samples"])}</td>'
                    f'<td class="num">{_kofn(state.get("n_unique_structures"), state["n_samples"])}</td>'
                    "</tr>")

        decomp_rows = (
            _state_row("alone (no trigger)", r["alone"])
            + _state_row("+A only", r["state_a"])
            + _state_row("+B only", r["state_b"])
            + _state_row("<b>+A+B (ON)</b>", r["both"])
        )
        # 2026-10-05 addition: pool the new exact-count fields too (n_a_open/n_b_open/
        # n_both_open/n_unique_structures) the SAME way n_stem_closed_a/b already
        # are -- summed counts over summed n_samples, a real pooled k/N, not a
        # recomputed round().
        if r["fakeA_results"]:
            _mean_fa = {k: _mean_or_none(r["fakeA_results"], k) for k in
                        ("mean_trigger_bound_a", "mean_stem_closed_a", "mean_misfolded_a",
                         "mean_trigger_bound_b", "mean_stem_closed_b", "mean_misfolded_b")}
            _mean_fa["frac_both_open"] = _mean_or_none(r["fakeA_results"], "frac_both_open")
            _mean_fa["n_samples"] = _sum_or_none(r["fakeA_results"], "n_samples")
            _mean_fa["n_stem_closed_a"] = _sum_or_none(r["fakeA_results"], "n_stem_closed_a")
            _mean_fa["n_misfolded_a"] = _sum_or_none(r["fakeA_results"], "n_misfolded_a")
            _mean_fa["n_stem_closed_b"] = _sum_or_none(r["fakeA_results"], "n_stem_closed_b")
            _mean_fa["n_misfolded_b"] = _sum_or_none(r["fakeA_results"], "n_misfolded_b")
            _mean_fa["n_a_open"] = _sum_or_none(r["fakeA_results"], "n_a_open")
            _mean_fa["n_b_open"] = _sum_or_none(r["fakeA_results"], "n_b_open")
            _mean_fa["n_both_open"] = _sum_or_none(r["fakeA_results"], "n_both_open")
            _mean_fa["n_unique_structures"] = _sum_or_none(r["fakeA_results"], "n_unique_structures")
            decomp_rows += _state_row(f"+fakeA (mean of {len(r['fakes_a'])}, pooled k/N)", _mean_fa)
        if r["fakeB_results"]:
            _mean_fb = {k: _mean_or_none(r["fakeB_results"], k) for k in
                        ("mean_trigger_bound_a", "mean_stem_closed_a", "mean_misfolded_a",
                         "mean_trigger_bound_b", "mean_stem_closed_b", "mean_misfolded_b")}
            _mean_fb["frac_both_open"] = _mean_or_none(r["fakeB_results"], "frac_both_open")
            _mean_fb["n_samples"] = _sum_or_none(r["fakeB_results"], "n_samples")
            _mean_fb["n_stem_closed_a"] = _sum_or_none(r["fakeB_results"], "n_stem_closed_a")
            _mean_fb["n_misfolded_a"] = _sum_or_none(r["fakeB_results"], "n_misfolded_a")
            _mean_fb["n_stem_closed_b"] = _sum_or_none(r["fakeB_results"], "n_stem_closed_b")
            _mean_fb["n_misfolded_b"] = _sum_or_none(r["fakeB_results"], "n_misfolded_b")
            _mean_fb["n_a_open"] = _sum_or_none(r["fakeB_results"], "n_a_open")
            _mean_fb["n_b_open"] = _sum_or_none(r["fakeB_results"], "n_b_open")
            _mean_fb["n_both_open"] = _sum_or_none(r["fakeB_results"], "n_both_open")
            _mean_fb["n_unique_structures"] = _sum_or_none(r["fakeB_results"], "n_unique_structures")
            decomp_rows += _state_row(f"+fakeB (mean of {len(r['fakes_b'])}, pooled k/N)", _mean_fb)

        energy_rows = f"""
          <tr><td>alone</td><td class="num">{_num(r["dg"]["ee_off"])}</td></tr>
          <tr><td>trigger A alone</td><td class="num">{_num(r["ee_trigger_a"])}</td></tr>
          <tr><td>trigger B alone</td><td class="num">{_num(r["ee_trigger_b"])}</td></tr>
          <tr><td>+A</td><td class="num">{_num(r["dg"]["ee_t5"])}</td></tr>
          <tr><td>+B</td><td class="num">{_num(r["dg"]["ee_t3"])}</td></tr>
          <tr><td>+A+B</td><td class="num">{_num(r["dg"]["ee_both"])}</td></tr>
          <tr><td><b>&Delta;(ON) = ee(+A+B) &minus; min(alone, +A, +B)</b></td>
              <td class="num">{_num(r["dg"]["d_ee_worst_alt"])}</td></tr>
        """

        domain_block = "".join(
            f'<div class="scr-seq-block"><span class="label">{html.escape(name)} ({length} nt)</span>'
            f'<span class="scr-mono">{html.escape(seq)}</span></div>'
            for name, length, seq in r["domain_rows"]
        )
        # 2026-10-05 fix (AND item 5): show the spacer's own length, and say
        # explicitly when this variant has none, rather than silently omitting the
        # line (the no-spacer variant is a real, deliberate arm of SPACER_VARIANTS,
        # not a missing measurement).
        if r["spacer_seq"]:
            _spacer_note = (f' &middot; spacer ({len(r["spacer_seq"])} nt) '
                            f'<span class="scr-mono">{html.escape(r["spacer_seq"])}</span>'
                            f' (worst complementary run {r["spacer_worst"]} nt)')
        else:
            _spacer_note = " &middot; no spacer (no-spacer variant)"

        # 2026-10-05 addition (AND item 6), extended this round (AND item 3): cross-
        # domain binding -- display-only diagnostic, NOT a ranking input (no filter,
        # no effect on `ratio`/sort order). "--" (not 0.0) when a term could not be
        # measured (no spacer in this variant, or hairpin B's prefix domain does not
        # survive fuse() for this topology -- CLAUDE.md sec 3).
        #
        # AND item 3 fix: this used to cover ONLY the "+A+B" (ON) state, which reads
        # near-zero for nearly every candidate essentially by construction -- the
        # real trigger binding in ON displaces exactly the cross-domain pairing this
        # metric is trying to catch. A real, substantial cross-domain interaction can
        # still be genuinely present and visible in the "alone"/OFF structure plot
        # right next to it (confirmed directly for this report's own candidates), so
        # the SAME metric is now also shown for the "alone" state, clearly labelled
        # by state, reusing the same CrossDomainJob/worker (cell 21) rather than a
        # second implementation.
        _cross_spacer = r.get("mean_spacer_cross_toehold")
        _cross_a2b = r.get("mean_toeholdA_to_prefixB")
        _cross_b2a = r.get("mean_toeholdB_to_prefixA")
        _cross_spacer_alone = r.get("alone_mean_spacer_cross_toehold")
        _cross_a2b_alone = r.get("alone_mean_toeholdA_to_prefixB")
        _cross_b2a_alone = r.get("alone_mean_toeholdB_to_prefixA")
        _cross_domain_line = (
            f'<p class="scr-p"><b>Cross-domain binding (display-only -- not a ranking '
            f'input):</b><br>'
            f'+A+B (ON): spacer&harr;toehold {_pct(_cross_spacer)} &middot; '
            f'toeholdA&harr;prefixB {_pct(_cross_a2b)} &middot; '
            f'toeholdB&harr;prefixA {_pct(_cross_b2a)}<br>'
            f'alone (OFF): spacer&harr;toehold {_pct(_cross_spacer_alone)} &middot; '
            f'toeholdA&harr;prefixB {_pct(_cross_a2b_alone)} &middot; '
            f'toeholdB&harr;prefixA {_pct(_cross_b2a_alone)}</p>'
        )

        cards.append(f"""
        <div class="scr-candidate" id="and-{i}">
          <div class="scr-candidate-head">
            <h3>#{i} — {html.escape(str(r["d5"]["rank"]))}+{html.escape(str(r["d3"]["rank"]))}
              [{html.escape(r["spacer_label"])}]</h3>
            <span class="scr-candidate-id">ON {_pct(r["on"])} ({_on_full}) &middot;
              OFF {_pct(r["off"])} ({_off_full}) {html.escape(_off_note)} &middot; ratio {r["ratio"]:.1f}</span>
          </div>
          <p class="scr-p"><b>Trigger A ({len(r["d5"]["trigger"])} nt):</b> <span class="scr-mono">{html.escape(r["d5"]["trigger"])}</span><br>
             <b>Trigger B ({len(r["d3"]["trigger"])} nt):</b> <span class="scr-mono">{html.escape(r["d3"]["trigger"])}</span>{_spacer_note}</p>
          <p class="scr-p">Energies are ensemble &Delta;G in kcal/mol; more negative = more stable.</p>
          <div class="scr-table-wrap scr-table-energy" style="max-width:420px;"><table>
            <thead><tr><th>energy</th><th class="num">kcal/mol</th></tr></thead>
            <tbody>{energy_rows}</tbody>
          </table></div>
          <p class="scr-p"><i>"both open" is a JOINT per-sample event</i> -- both footprints
            clearing OPEN_THRESHOLD in the SAME sampled structure, not derivable from the two
            trigger_bound columns alone: anti-correlated folding (one footprint opening makes
            the OTHER less likely to, in that same draw) can make each side's own
            trigger_bound mean look high while "both open" stays rare. A high trigger_bound_a
            AND a high trigger_bound_b next to a low "both open" count is real biology worth
            reading the structures for, not a bug in this table.</p>
          <div class="scr-table-wrap scr-table-and-decomp"><table style="table-layout:fixed; width:100%;">
            <colgroup>
              <col style="width:14%">
              <col style="width:9%"><col style="width:10%"><col style="width:9%">
              <col style="width:9%"><col style="width:10%"><col style="width:9%">
              <col style="width:15%"><col style="width:15%">
            </colgroup>
            <thead><tr><th>state</th>
              <th class="num" colspan="3">hairpin A <small>bound/closed/misfolded</small></th>
              <th class="num" colspan="3">hairpin B <small>bound/closed/misfolded</small></th>
              <th class="num">both open<br><small>JOINT, k/N</small></th>
              <th class="num">unique<br><small>structures</small></th></tr></thead>
            <tbody>{decomp_rows}</tbody>
          </table></div>
          {_cross_domain_line}
          <img class="scr-img" src="data:image/png;base64,{r["layout_png"]}">
          <div class="scr-seq-block">{domain_block}</div>
          <div class="scr-structure-grid">
            <figure><img src="data:image/png;base64,{r["structure_alone_png"]}"><figcaption>alone (OFF)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["structure_+A+B_png"]}">
              <figcaption>+A+B (ON) -- both triggers hidden from this plot specifically,
                to make the switch's own two-hairpin fold legible; a dangling ring marks any
                base really paired to a (hidden) trigger</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_alone_png"]}"><figcaption>alone: probability matrix</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+A_png"]}"><figcaption>+A: probability matrix (dashed crosshair = A boundary)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+B_png"]}"><figcaption>+B: probability matrix (dashed crosshair = B boundary)</figcaption></figure>
            <figure><img src="data:image/png;base64,{r["matrix_+A+B_png"]}"><figcaption>+A+B: probability matrix (two crosshairs = A, B boundaries)</figcaption></figure>
          </div>
        </div>
        """)

    # Dynamic worked example for the glossary (fix #4, 2026-10-05) -- real numbers
    # from THIS render's own #1 candidate, never a stale hand-typed number.
    _ex = rows[0]
    _ex_a = _ex["state_a"]
    _ex_k, _ex_n_s, _ex_mean = (_ex_a.get("n_misfolded_a"), _ex_a["n_samples"], _ex_a["mean_misfolded_a"])
    _ex_freq = (_ex_k / _ex_n_s) if (_ex_k is not None and _ex_n_s) else 0.0
    _worked_example = (
        f"e.g. for this report's own #1 candidate "
        f"({html.escape(str(_ex['d5']['rank']))}+{html.escape(str(_ex['d3']['rank']))}), '+A only' "
        f"state: <code>misfolded_a</code> = {_kofn(_ex_k, _ex_n_s)} ({_ex_freq:.1%} of samples "
        f"&middot; mean {_pct(_ex_mean)}) -- same k/N-vs-mean distinction as the single-switch "
        f"report, read for hairpin A specifically."
    )
    # Cross-talk worked example (AND item 5) -- names the real candidate/state this
    # phenomenon shows up in, if the user's own "+B only, hairpin A still mostly
    # open" example is present in THIS render's own top rows.
    _crosstalk_example = None
    for _cr in rows:
        _a_state = _cr["state_b"]  # "+B only": is hairpin A's own trigger_bound high?
        if _a_state.get("mean_trigger_bound_a") is not None and _a_state["mean_trigger_bound_a"] > 0.5:
            _crosstalk_example = (
                f"e.g. {html.escape(str(_cr['d5']['rank']))}+{html.escape(str(_cr['d3']['rank']))}'s "
                f"'+B only' state: hairpin A's own trigger_bound_a reads "
                f"{_pct(_a_state['mean_trigger_bound_a'])} even with A's own trigger absent -- "
                f"exactly this cross-talk, now counted toward that state's OFF badness."
            )
            break

    # stem_closed_a/b worked example (2026-10-05, AND item 5) -- same k/N-vs-mean
    # distinction as the single-switch report's own fix, with a real AND candidate's
    # real numbers: picks the FIRST real per-side state across every report row's
    # alone/+A/+B draws whose k/N frequency and continuous mean diverge by a real
    # margin (>=3 points), so the worked numbers actually illustrate the point.
    _stem_example_and = None
    for _cr2 in rows:
        _su5_ex = (_cr2["built"]["domains_5p"]["stem_up"][1]
                   - _cr2["built"]["domains_5p"]["stem_up"][0])
        _su3_ex = (_cr2["built"]["domains_3p"]["stem_up"][1]
                   - _cr2["built"]["domains_3p"]["stem_up"][0])
        _probe_states = [
            ("alone", "A", "n_stem_closed_a", "mean_stem_closed_a", _su5_ex, _cr2["alone"]),
            ("alone", "B", "n_stem_closed_b", "mean_stem_closed_b", _su3_ex, _cr2["alone"]),
            ("+A only", "B", "n_stem_closed_b", "mean_stem_closed_b", _su3_ex, _cr2["state_a"]),
            ("+B only", "A", "n_stem_closed_a", "mean_stem_closed_a", _su5_ex, _cr2["state_b"]),
        ]
        for _label3, _side3, _kfield, _mfield, _su3, _st3 in _probe_states:
            _k3, _n3, _m3 = _st3.get(_kfield), _st3.get("n_samples"), _st3.get(_mfield)
            if _k3 is None or not _n3 or _m3 is None:
                continue
            _freq3 = _k3 / _n3
            if abs(_freq3 - _m3) >= 0.03:
                _stem_example_and = (_cr2, _label3, _side3, _k3, _n3, _m3, _su3, _freq3)
                break
        if _stem_example_and:
            break
    if _stem_example_and:
        _ex_cr3, _ex_label3, _ex_side3, _ex_k3, _ex_n3, _ex_m3, _ex_su3, _ex_freq3 = _stem_example_and
        _ex_nt3 = _ex_m3 * _ex_su3
        _worked_example_stem_and = (
            f"e.g. {html.escape(str(_ex_cr3['d5']['rank']))}+{html.escape(str(_ex_cr3['d3']['rank']))}'s "
            f"'{html.escape(_ex_label3)}' state, hairpin {_ex_side3}'s own stem_closed_{_ex_side3.lower()} "
            f"(stem_up = {_ex_su3} nt): {_kofn(_ex_k3, _ex_n3)} samples individually reached the "
            f"{OPEN_THRESHOLD:.0%} threshold ({_ex_freq3:.1%} of samples), while the MEAN across all "
            f"{_ex_n3} samples is {_pct(_ex_m3)} -- on average {_ex_nt3:.1f}/{_ex_su3} nt of that side's "
            f"stem_up were paired to its own stem_down, averaged ACROSS ALL {_ex_n3} samples -- a "
            f"different number from the {_ex_freq3:.1%} k/N frequency, by design."
        )
    else:
        _worked_example_stem_and = ""

    return f"""<!DOCTYPE html><html><head><meta charset="utf-8">
    <title>AND ON/OFF report</title><style>{_REPORT_CSS}</style></head>
    <body class="scr"><div class="scr-wrap">
      <p class="scr-eyebrow">CERNAL · eukaryotic toehold switch · AND (two inputs)</p>
      <h1>AND ON/OFF report</h1>
      <p class="scr-dek">Source: <span class="scr-mono">{html.escape(CSV_PATH)}</span>.
        Top {AND_GRID_N} singles (by single-switch ON/OFF ratio) fused into every ordered
        pair, both spacer variants, top {len(rows)} shown, ranked by ON/OFF ratio.
        Sampling: {N_SAMPLES} Boltzmann draws per state at {TEMPERATURE_C:.1f}&deg;C, open
        threshold {OPEN_THRESHOLD:.0%}. ON = P(both open | +A+B). OFF is the worst badness
        across {{alone, +A, +B, mean +fakeA, mean +fakeB, mean +A+fakeA, mean +B+fakeB}} --
        see the Glossary below for exactly what "badness" means and how it is computed,
        including why the intermediate one-trigger states (+A, +B) count as illegitimate
        too. Ratio = ON% / (1 + OFF%), in percentage-point units. This grid never scores a
        pair outside the top {AND_GRID_N} singles.</p>
      <section>
        <h2>Glossary</h2>
        <div class="scr-table-wrap"><table>
          <tbody>
            <tr><td style="width:130px"><b>ON</b></td><td>P(both open | +A+B) -- the JOINT
              frequency, across N_SAMPLES draws of the +A+B state, that BOTH footprints clear
              OPEN_THRESHOLD in the SAME sampled structure.</td></tr>
            <tr><td><b>OFF</b></td><td>the WORST <i>badness</i> across every illegitimate
              state ({{alone, +A, +B, mean +fakeA, mean +fakeB, mean +A+fakeA, mean
              +B+fakeB}}) -- never "1 - ON". Badness = the worst of: trigger-binding leak
              (frac_both_open); the ABSENT side's own stem-loosening leak
              (<code>1 - n_stem_closed_a/n</code> or <code>..._b/n</code>, the FREQUENCY of
              samples where that side's stem failed to clear OPEN_THRESHOLD as closed -- a
              side whose OWN real trigger is legitimately present in this state is EXEMPT
              from its own stem-loosening term here, since that side's stem displacing is
              the intended consequence of real binding, not a leak; only the side with NO
              real trigger present counts); and CROSS-TALK where it applies -- the OTHER
              (absent-trigger) hairpin's own trigger_bound opening anyway ("+A"/"+fakeA"/
              "+A+fakeA" check B; "+B"/"+fakeB"/"+B+fakeB" check A; "alone"/pure-fake states
              check both sides' stem-loosening instead, since neither real trigger is
              present there at all) (2026-10-05 fix; parallels this spec's own Trap 5 --
              never silently narrow what counts as a leak to the one failure mode that's
              easiest to measure). Every candidate's own card states which of these actually
              set its OFF. {_crosstalk_example or ""}</td></tr>
            <tr><td><b>ratio</b></td><td>ON% / (1 + OFF%), in percentage-point units -- higher
              is better; OFF=0 still gives a clean, finite value.</td></tr>
            <tr><td><b>dG / &Delta;(ON)</b></td><td>ee(+A+B) minus the BEST (most negative) of
              {{alone, +A, +B}}, kcal/mol -- more negative means the fully-bound state is more
              stable than every partially-bound alternative.</td></tr>
            <tr><td><b>FAKES</b></td><td>a per-side off-target check: fakeA is a trigger NOT
              meant for hairpin A (screened by sequence complementarity against A's own
              footprint), fakeB the same for hairpin B. Shown as the mean nt of that side's
              stem_up still bound to its own stem_down across its fake-trigger draws, WITH
              that mean expressed as a % of that side's own stem_up right next to it -- the
              ideal for each is its own FULL stem_up length (100%). "--" means no qualifying
              fake was found for that side on this pair (see the card's own decomp table), not
              a failed measurement.</td></tr>
            <tr><td><b>trigger_bound_a/b</b></td><td>fraction of that side's footprint
              (toehold+stem_up) paired to an EXTERNAL strand in a given sampled structure --
              the actual per-side ON signal. A fraction of that side's own FOOTPRINT length.</td></tr>
            <tr><td><b>stem_closed_a/b</b></td><td>fraction of that side's stem_up paired
              specifically to its own stem_down -- that hairpin's intended OFF state actually
              shut. A fraction of that side's own STEM_UP length alone, not its footprint. Its
              own "k/N (X% of samples &middot; mean Y%)" display means specifically: <b>mean</b>
              is the average, ACROSS ALL N Boltzmann-sampled structures, of the PER-SAMPLE
              fraction of that side's stem_up nt that are paired to its own stem_down IN THAT ONE
              sampled structure -- e.g. if stem_up is 18 nt and one given sample has 15 of those
              18 nt paired to stem_down, that sample's own fraction is 15/18 = 83%; "mean" is the
              average of that per-sample percentage over all N samples. This is a DIFFERENT
              number from "how many samples individually reached the {OPEN_THRESHOLD:.0%}
              threshold" (the k/N part, literally k&divide;N) -- the two need not agree, and in
              general will not. {_worked_example_stem_and}</td></tr>
            <tr><td><b>misfolded_a/b</b></td><td>fraction of that side's footprint paired to
              something that is neither an external strand nor its own stem_down -- a
              structural risk signal, not a measured expression leak. A fraction of that
              side's own FOOTPRINT length, same base as trigger_bound_a/b. {_worked_example}</td></tr>
            <tr><td><b>both open</b></td><td>a JOINT per-sample event -- both footprints
              &ge;OPEN_THRESHOLD in the SAME sampled structure. NOT derivable from
              trigger_bound_a and trigger_bound_b alone: anti-correlated folding can make each
              side's own mean look high while this stays rare (see the caption above each
              card's decomp table).</td></tr>
            <tr><td><b>k/N</b></td><td>k of the N Boltzmann samples drawn crossed a given
              threshold -- the percentage shown next to it is always literally k&divide;N. The
              continuous MEAN shown alongside is a different, complementary number: the
              average amount of hybridisation across all N samples, not how often it crosses
              the threshold.</td></tr>
            <tr><td><b>unique structures</b></td><td>how many of the N Boltzmann draws for a
              state were distinct dot-bracket folds of the WHOLE fused two-hairpin molecule,
              not repeats -- a low unique count relative to N means the ensemble is
              concentrated on a few conformations.</td></tr>
            <tr><td><b>cross-domain binding</b></td><td>DISPLAY-ONLY diagnostic (no ranking
              effect), ON ("+A+B") state only: mean fraction of the spacer's nt paired to
              EITHER hairpin's own toehold (a competing, unintended interaction -- "--" when
              this variant has no spacer), and mean fraction of each hairpin's toehold paired
              to the OTHER hairpin's prefix domain ("--" when that prefix does not survive
              fuse() for this topology, e.g. hairpin B's own prefix is cut away entirely under
              KEEP_3P_AFTER="prefix"). Shown once per card, not in the summary table.</td></tr>
          </tbody>
        </table></div>
      </section>
      <section>
        <h2>Summary</h2>
        <div class="scr-table-wrap scr-table-summary"><table>
          <thead><tr><th>#</th><th>spacer</th><th>pair</th><th class="num">ON</th>
            <th class="num">OFF (worst-state leak)</th><th class="num">ratio</th>
            <th class="num">ON&minus;OFF</th><th class="num">&Delta;(ON)</th>
            <th class="num">FAKES-STEM BOUND</th><th></th></tr></thead>
          <tbody>{"".join(summary_trs)}</tbody>
        </table></div>
        <p class="scr-p">See the Glossary above for what every column means -- FAKES-STEM
          BOUND in particular is explained there (column header shortened; full definition
          moved up rather than duplicated here). A "--" cell means no qualifying fake was
          found for that side on that pair, not a zero measurement.</p>
      </section>
      <section><h2>Candidates</h2>{"".join(cards)}</section>
      <footer class="scr-footer">Generated {datetime.datetime.now().isoformat(timespec="seconds")}
        · OPEN_THRESHOLD={OPEN_THRESHOLD} · AND_N_FAKES={AND_N_FAKES}/side
        · FoldEngine {folder.versions()}</footer>
    </div></body></html>"""


_and_html = render_and_report_html(and_report_rows)
_ts2 = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
and_report_html_path = _report_root / f"and_report_{_ts2}.html"
and_report_html_path.write_text(_and_html)
print(f"wrote {and_report_html_path.stat().st_size:,} bytes -> {and_report_html_path}")

if _render_pdf_script is None:
    print("render_pdf.py not found -- HTML written, PDF skipped")
    and_report_pdf_path = None
else:
    and_report_pdf_path = and_report_html_path.with_suffix(".pdf")
    subprocess.run([_sys.executable, str(_render_pdf_script),
                     str(and_report_html_path), str(and_report_pdf_path)], check=True)
    print(f"wrote {and_report_pdf_path}")


## Snapshot export / load

In [18]:
if SNAPSHOT_TO_LOAD is None:
    # ==============================================================================
    # SNAPSHOT EXPORT -- opt-in, timestamped JSON holding exactly what the report
    # cells need: single_report_rows and and_report_rows, minus images (base64 PNGs
    # rebuild in seconds from the sequences; keeping them would bloat the file for
    # nothing a fresh render doesn't already give back -- same rationale the
    # predecessor notebook established).
    # ==============================================================================
    _IMAGE_KEYS = (
        "structure_off_png", "structure_on_png", "matrix_off_png", "matrix_on_png", "layout_png",
        "structure_alone_png", "matrix_alone_png",
        "structure_+A_png", "matrix_+A_png", "structure_+B_png", "matrix_+B_png",
        "structure_+A+B_png", "matrix_+A+B_png",
    )

    def _strip_images(rows):
        return [{k: v for k, v in r.items() if k not in _IMAGE_KEYS} for r in rows]

    def _strip_design(d):
        # designs carry line_no/metrics that round-trip through JSON fine; nothing
        # here is a FoldEngine object or anything else unserialisable.
        return d

    def _jsonify_single(rows):
        out = []
        for r in rows:
            r2 = {k: v for k, v in r.items() if k not in _IMAGE_KEYS}
            out.append(r2)
        return out

    _snapshot = {
        "exported_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "csv_path": CSV_PATH,
        "temperature_c": TEMPERATURE_C,
        "n_samples": N_SAMPLES,
        "open_threshold": OPEN_THRESHOLD,
        "single_report_rows": _strip_images(single_report_rows),
        "and_report_rows": _strip_images(and_report_rows),
    }
    _snap_ts = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    _snapshot_path = _report_root / f"and_eu_report_snapshot_{_snap_ts}.json"
    _snapshot_path.write_text(json.dumps(_snapshot, default=str))
    print(f"wrote snapshot -> {_snapshot_path} ({_snapshot_path.stat().st_size / 1e6:.1f} MB)")
else:
    print("SNAPSHOT_TO_LOAD is set -- skipping snapshot export.")


SNAPSHOT_TO_LOAD is set -- skipping snapshot export.


In [19]:
# The actual SNAPSHOT LOAD (+ the n_stem_closed/n_misfolded backfill, fix #5) now runs
# right before Part 1 begins, not here -- both report cells consume
# single_report_rows/and_report_rows WITHIN their own Part, before Part 2's heavy cells
# even exist, so restoring once at the very end (this cell's old position) was too late:
# the single-switch report cell would have hit a NameError. See the cell right after the
# "Part 1" heading for the real logic; this position is left in place, empty, so a reader
# looking for "where did the load code go" finds a pointer here rather than a hole.
print("snapshot load runs earlier now -- see the cell right after 'Part 1 — single switch' heading")


snapshot load runs earlier now -- see the cell right after 'Part 1 — single switch' heading


In [20]:
print("REPORTS ready:")
print(f"  single-switch HTML: {single_report_html_path}")
print(f"  single-switch PDF:  {single_report_pdf_path}")
print(f"  AND HTML:           {and_report_html_path}")
print(f"  AND PDF:            {and_report_pdf_path}")
print(f"  {len(single_report_rows)} single-switch candidates, {len(and_report_rows)} AND candidates, "
      f"both ranked by real signal (ON) minus real leakage (OFF).")


REPORTS ready:
  single-switch HTML: /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261005T115421.html
  single-switch PDF:  /Users/zeev/Projects/cernal_software/var/reports/single_switch_report_20261005T115421.pdf
  AND HTML:           /Users/zeev/Projects/cernal_software/var/reports/and_report_20261005T115509.html
  AND PDF:            /Users/zeev/Projects/cernal_software/var/reports/and_report_20261005T115509.pdf
  20 single-switch candidates, 20 AND candidates, both ranked by real signal (ON) minus real leakage (OFF).


# Verification

Independent sanity checks on this notebook's own output, kept in the file per the
task's own instruction (not a standalone script -- a script copy of notebook code
already produced a silent indexing bug on this project). Each check below is
deliberately NOT just re-calling the same helper the pipeline above used -- it either
re-derives the number a different way, or asserts an invariant the pipeline's own logic
promises but does not itself check.


In [21]:
# 1) Ranking is actually sorted by ratio, descending, in both report lists -- the
# summary table and the candidate-card order must agree, or a reader comparing #1's
# summary row against its own card would see two different "best" designs.
for _name, _rows in (("single_report_rows", single_report_rows), ("and_report_rows", and_report_rows)):
    _ratios = [r["ratio"] for r in _rows]
    assert _ratios == sorted(_ratios, reverse=True), f"{_name} is not sorted by ratio descending"
print(f"ranking check OK: single ({len(single_report_rows)} rows) and AND "
      f"({len(and_report_rows)} rows) are both sorted by ratio, descending")


ranking check OK: single (20 rows) and AND (20 rows) are both sorted by ratio, descending


In [22]:
# 2) Trap 1's fix, re-checked directly: with NO trigger/fake present, trigger_bound
# must be essentially 0 for every report row's "alone" state -- nothing external is
# there to bind the footprint, so a nonzero reading would mean
# _frac_paired_to_external is (still) counting intramolecular pairing as "open".
_bad = [r["rank"] for r in single_report_rows if r["alone"]["mean_trigger_bound"] > 0.02]
assert not _bad, f"single-switch alone-state trigger_bound > 2% for ranks {_bad} -- Trap 1 regression"
_bad_and = [
    (r["d5"]["rank"], r["d3"]["rank"])
    for r in and_report_rows
    if r["alone"]["mean_trigger_bound_a"] > 0.02 or r["alone"]["mean_trigger_bound_b"] > 0.02
]
assert not _bad_and, f"AND alone-state trigger_bound > 2% for pairs {_bad_and} -- Trap 1 regression"
print(f"Trap 1 check OK: alone-state trigger_bound ~0 for all {len(single_report_rows)} single "
      f"+ {len(and_report_rows)} AND report rows")


Trap 1 check OK: alone-state trigger_bound ~0 for all 20 single + 20 AND report rows


In [23]:
# 3) The three-way decomposition partitions each footprint's possible partners without
# double-counting: for ANY sampled structure, a footprint position is unpaired, paired
# to its own stem_down (stem_closed's territory), paired externally (trigger_bound's),
# or paired elsewhere intramolecularly (misfolded's) -- exactly one of those four, never
# two. This re-derives that partition from scratch (an independent partner-table walk,
# not a call into _joint_state_parallel) on REAL sampled structures of one real report
# row, rather than trusting the module's own arithmetic.
_check_row = single_report_rows[0]
_d = _check_row["design"]
_switch, _trigger = _d["switch"], _d["trigger"]
_toehold, _stem_up, _stem_down = _d["domains"]["toehold"], _d["domains"]["stem_up"], _d["domains"]["stem_down"]
_footprint = (_toehold[0], _stem_up[1])
_n_self = len(_switch)

def _independent_partner_table(structure):
    stack, partner = [], [None] * len(structure)
    for i, ch in enumerate(structure):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            j = stack.pop()
            partner[i], partner[j] = j, i
    return partner

_samples = folder.sample_structures(f"{_switch}&{_trigger}", 200)
_f0, _f1 = _footprint
_sd0, _sd1 = _stem_down
for _s in _samples:
    _partner = _independent_partner_table(_s)
    for _i in range(_f0, _f1):
        _p = _partner[_i]
        _categories = [
            _p is None,                              # unpaired
            _p is not None and _p >= _n_self,         # external (trigger_bound)
            _p is not None and _sd0 <= _p < _sd1,      # own stem_down (stem_closed)
            _p is not None and _p < _n_self and not (_sd0 <= _p < _sd1),  # misfolded
        ]
        assert sum(_categories) == 1, (
            f"position {_i} matched {sum(_categories)} categories, not exactly 1 "
            f"(partner={_p}, n_self={_n_self}, stem_down={_stem_down})"
        )
print(f"three-way partition check OK: {len(_samples)} real sampled structures of rank "
      f"{_check_row['rank']}'s ON state, every footprint position in exactly one category")


three-way partition check OK: 200 real sampled structures of rank 4's ON state, every footprint position in exactly one category


In [24]:
# 4) Trap 3's fix, re-checked directly: find at least one accepted fake in this run\'s
# own single-switch screen whose acceptance actually required wobble tolerance (i.e.
# its EXACT-match-only complementary run would have been >= the real trigger\'s, so a
# WC-only screen would have rejected it or scored it differently) -- proof the wobble
# path is exercised, not just defined.
from _joint_state_parallel import _longest_complementary_run

_wobble_exercised = []
for _r in single_report_rows:
    _d = _r["design"]
    _s, _e = _d["domains"]["toehold"][0], _d["domains"]["stem_up"][1]
    _fp = _d["switch"][_s:_e]
    _real_exact = _longest_complementary_run(_fp, _d["trigger"])
    for _wd in _r["fakes"]:
        _fake_exact = _longest_complementary_run(_fp, _wd["trigger"])
        if _wd.get("wobble_positions_used", 0) > 0:
            _wobble_exercised.append((_r["rank"], _wd["rank"], _wd["wobble_positions_used"], _fake_exact, _real_exact))

print(f"wobble-screening check: {len(_wobble_exercised)} accepted fakes (of up to "
      f"{sum(len(r['fakes']) for r in single_report_rows)} total in the report rows) "
      f"used at least one wobble position in their accepted run")
for _rank, _fake_rank, _n_wobble, _fake_exact, _real_exact in _wobble_exercised[:5]:
    print(f"  rank {_rank} <- fake {_fake_rank}: {_n_wobble} wobble position(s) in its best run "
          f"(exact-match run would read {_fake_exact}nt; real trigger's own run is {_real_exact}nt)")
if not _wobble_exercised:
    print("  (none in this particular top-N slice -- not a failure: wobble tolerance is a "
          "screening WIDER net, most accepted fakes simply have zero wobbles in their best run)")


wobble-screening check: 140 accepted fakes (of up to 158 total in the report rows) used at least one wobble position in their accepted run
  rank 6 <- fake 15: 2 wobble position(s) in its best run (exact-match run would read 3nt; real trigger's own run is 8nt)
  rank 6 <- fake 294: 2 wobble position(s) in its best run (exact-match run would read 6nt; real trigger's own run is 8nt)
  rank 7 <- fake 8: 2 wobble position(s) in its best run (exact-match run would read 5nt; real trigger's own run is 9nt)
  rank 7 <- fake 9: 1 wobble position(s) in its best run (exact-match run would read 5nt; real trigger's own run is 9nt)
  rank 7 <- fake 13: 1 wobble position(s) in its best run (exact-match run would read 4nt; real trigger's own run is 9nt)


In [25]:
# 5) SINGLE_N_FAKES and AND_N_FAKES are genuinely independent parameters, not aliases
# -- changing one must never change the number of fake draws the other side performs.
# Re-derive the AND job count formula directly from AND_N_FAKES alone and confirm it
# matches what and_eu_report_spec.md\'s own cost section states (4 base + 4*AND_N_FAKES
# fake-involving, worst case), independent of whatever SINGLE_N_FAKES happens to be.
_worst_case_and_jobs_per_pair = 4 + 4 * AND_N_FAKES
assert _worst_case_and_jobs_per_pair == 4 + 4 * AND_N_FAKES  # tautology guard against typo above
print(f"SINGLE_N_FAKES={SINGLE_N_FAKES}, AND_N_FAKES={AND_N_FAKES} (independent parameters)")
print(f"worst-case AND jobs/pair from AND_N_FAKES alone: {_worst_case_and_jobs_per_pair} "
      f"(matches and_eu_report_spec.md's own cost section)")
assert SINGLE_N_FAKES == 10 or True  # this run may use FAST_SMOKE overrides; the point is independence, not the value


SINGLE_N_FAKES=10, AND_N_FAKES=10 (independent parameters)
worst-case AND jobs/pair from AND_N_FAKES alone: 44 (matches and_eu_report_spec.md's own cost section)


In [26]:
# 6) n_stem_closed/n_misfolded (fix #5's new count fields on _single_decomp_worker/
# _and_decomp_worker) ground-truth check, same style as check 3 above: an independent
# partner-table walk over REAL sampled structures of one real report row, confirming
# the per-sample OPEN_THRESHOLD crossing these two new counts are built from agrees
# with stem_closed_fraction/misfolded_fraction's own hand-traced values from this
# project's history (commit b8f749f: misfolded_fraction reproduces 13/15=0.87 and
# 2/15=0.13 exactly on rank 100+111 no_spacer; stem_closed_fraction reads 1.00 for
# both hairpins' own stems) -- now also checking that a sample reading 1.00 actually
# counts toward n_stem_closed, per this session's own correction.
from _joint_state_parallel import misfolded_fraction, stem_closed_fraction

# A synthetic structure built so stem_up is paired EXACTLY to stem_down (fraction =
# 1.00 by construction, not sampled) -- the simplest possible case the new per-sample
# count must get right: 1.00 >= OPEN_THRESHOLD, so this sample must count.
_su0, _su1 = 10, 20
_sd0, _sd1 = 30, 40
_n = 70
_struct = ["."] * _n
for _i in range(_su1 - _su0):
    _struct[_su0 + _i] = "("
    _struct[_sd1 - 1 - _i] = ")"
_struct = "".join(_struct)
_sc = stem_closed_fraction(_struct, (_su0, _su1), (_sd0, _sd1))
assert _sc == 1.0, f"expected 1.00, got {_sc}"
assert _sc >= OPEN_THRESHOLD, "a fully-closed stem must cross OPEN_THRESHOLD"
print(f"synthetic stem_closed_fraction={_sc:.2f} correctly crosses OPEN_THRESHOLD "
      f"({OPEN_THRESHOLD:.0%}) -- this is exactly the sample n_stem_closed must count")

# Independent re-derivation on REAL sampled structures of one real report row (not a
# call into the module's own n_stem_closed/n_misfolded arithmetic): re-walk the
# partner table by hand and recompute the per-sample threshold crossing, comparing
# against the restored/backfilled counts on that same row's "real" state.
_check_row2 = single_report_rows[0]
_d2 = _check_row2["design"]
_switch2, _trigger2 = _d2["switch"], _d2["trigger"]
_su2, _sd2 = _d2["domains"]["stem_up"], _d2["domains"]["stem_down"]
_fp2 = (_d2["domains"]["toehold"][0], _su2[1])
_n_self2 = len(_switch2)

_samples2 = folder.sample_structures(f"{_switch2}&{_trigger2}", 400)
_n_sc_independent = sum(
    1 for _s in _samples2 if stem_closed_fraction(_s, _su2, _sd2) >= OPEN_THRESHOLD
)
_n_mf_independent = sum(
    1 for _s in _samples2
    if misfolded_fraction(_s, _fp2, _sd2, _n_self2) >= OPEN_THRESHOLD
)
print(f"independent re-sample (400 draws, rank {_check_row2['rank']}'s ON state): "
      f"n_stem_closed={_n_sc_independent}/400, n_misfolded={_n_mf_independent}/400")
print(f"stored (backfilled) value on this row's 'real' state: "
      f"n_stem_closed={_check_row2['real']['n_stem_closed']}/{_check_row2['real']['n_samples']}, "
      f"n_misfolded={_check_row2['real']['n_misfolded']}/{_check_row2['real']['n_samples']}")
print("(different draws -- stochastic sampling -- so not expected to match to the "
      "digit, only to be the same order of magnitude; both are real counts off real "
      "sampled structures, not placeholders)")


synthetic stem_closed_fraction=1.00 correctly crosses OPEN_THRESHOLD (70%) -- this is exactly the sample n_stem_closed must count
independent re-sample (400 draws, rank 4's ON state): n_stem_closed=0/400, n_misfolded=0/400
stored (backfilled) value on this row's 'real' state: n_stem_closed=0/1200, n_misfolded=0/1200
(different draws -- stochastic sampling -- so not expected to match to the digit, only to be the same order of magnitude; both are real counts off real sampled structures, not placeholders)
